In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, math, copy, random, time, threading, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [ ]:
DATA_PATH = "/content/drive/MyDrive/298/Elec/Data/LD2011_2014.txt"

PROJECT_DIR    = "/content/drive/MyDrive/298/Elec/SanityCheck/fgsm"
BASE_MODEL_DIR = "/content/drive/MyDrive/298/Elec/SanityCheck/models/baseModel"

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(BASE_MODEL_DIR, exist_ok=True)

STUDENTS_DIR     = os.path.join(PROJECT_DIR, "students")
STUDENTS_ADV_DIR = os.path.join(PROJECT_DIR, "students_adv")
RESULTS_DIR      = os.path.join(PROJECT_DIR, "results")

for d in [STUDENTS_DIR, STUDENTS_ADV_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

In [ ]:
FREQ = "15min"
STEPS_PER_HOUR = 4
HOURS_PER_DAY  = 24

HIST_HOURS  = 96
HZN_HOURS   = 24
HIST_STEPS  = HIST_HOURS * STEPS_PER_HOUR   # 384
HZN_STEPS   = HZN_HOURS  * STEPS_PER_HOUR   # 96
SLIDE       = 1

TRAIN_FRAC  = 0.70
VAL_FRAC    = 0.10

BATCH       = 16
PATCH       = 4
D_FEATS     = 1
D_MODEL     = 256
N_HEAD      = 4
N_LAYERS    = 3
DIM_FF      = 512
DROPOUT     = 0.1

LR_BASE     = 5e-4
LR_STUDENT  = 1e-4
EPOCHS_BASE = 30
EPOCHS_STU  = 5

In [ ]:
FGSM_TRAIN_EPS_LIST = [0.1, 0.2, 0.3, 0.5]

RFGSM_EVAL_EPS_LIST = [0.1, 0.2, 0.3, 0.5]
RFGSM_ALPHA_FACTOR  = 1.0

N_RUNS = 3

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", device)

Device: cuda


In [ ]:
HEARTBEAT_SECS = 600  # 10 minutes

def start_heartbeat(name="task"):
    flag = {"on": True}
    start_time = time.time()

    def _hb():
        while flag["on"]:
            time.sleep(HEARTBEAT_SECS)
            if flag["on"]:
                elapsed = (time.time() - start_time) / 60.0
                print(f"[heartbeat:{name}] running... elapsed={elapsed:.2f} min", flush=True)

    t = threading.Thread(target=_hb, daemon=True)
    t.start()
    return flag, t

def stop_heartbeat(flag, thread):
    flag["on"] = False
    try:
        thread.join(timeout=1)
    except Exception:
        pass

In [ ]:
print("Reading raw txt:", DATA_PATH)
df = pd.read_csv(
    DATA_PATH,
    sep=';',
    quotechar='"',
    decimal=',',
    parse_dates=[0],
    index_col=0,
    na_values=['', 'NA', 'NaN'],
    low_memory=False
)
df.index.name = "timestamp"
df = df.sort_index()
print("Raw shape:", df.shape)

METER = "MT_001"
print("Using single meter:", METER)

# restrict to 2013 only
START_DT = pd.Timestamp("2013-01-01 00:00:00")
END_DT   = pd.Timestamp("2013-12-31 23:45:00")

df = df[[METER]]
df = df.loc[(df.index >= START_DT) & (df.index <= END_DT)].copy()

Reading raw txt: /content/drive/MyDrive/298/Elec/Data/LD2011_2014.txt
Raw shape: (140256, 370)
Using single meter: MT_001


In [ ]:
# regularize index to strict 15 min grid

full_idx = pd.date_range(START_DT, END_DT, freq=FREQ)
df = df.reindex(full_idx)

In [ ]:
# fill up to 4 consecutive missing steps forward & backward

df = df.apply(pd.to_numeric, errors='coerce')
df = df.ffill(limit=4).bfill(limit=4)
print("Post filtering shape:", df.shape)

Post filtering shape: (35040, 1)


In [ ]:
# scale on early TRAIN_FRAC portion of time
n_t       = len(df)
split_idx = int(TRAIN_FRAC * n_t)
df_train  = df.iloc[:split_idx]
df_all    = df

scaler = MinMaxScaler()
train_scaled = pd.DataFrame(
    scaler.fit_transform(df_train),
    index=df_train.index,
    columns=df_train.columns
)
full_scaled = pd.DataFrame(
    scaler.transform(df_all),
    index=df_all.index,
    columns=df_all.columns
)
print("Scaled shape:", full_scaled.shape)

Scaled shape: (35040, 1)


In [ ]:
# windowing

values = full_scaled.values.astype(np.float32)  # (T, 1)
T, D = values.shape
assert D == 1, f"Expected 1 feature, got {D}"
print("T, D:", T, D)

X_list, Y_list, meta = [], [], []
for t in range(HIST_STEPS, T - HZN_STEPS, SLIDE):
    x = values[t - HIST_STEPS:t, :]      # (384, 1)
    y = values[t:t + HZN_STEPS, :]       # (96, 1)
    X_list.append(x)
    Y_list.append(y)
    meta.append((
        full_scaled.index[t - HIST_STEPS],
        full_scaled.index[t - 1],
        full_scaled.index[t],
        full_scaled.index[t + HZN_STEPS - 1],
    ))

X = np.stack(X_list)   # (N, 384, 1)
Y = np.stack(Y_list)   # (N, 96, 1)
meta_df = pd.DataFrame(meta, columns=["x_start", "x_end", "y_start", "y_end"])
print("Windowed tensors:", X.shape, Y.shape)

T, D: 35040 1
Windowed tensors: (34560, 384, 1) (34560, 96, 1)


In [ ]:
# chronological split

N = len(X)
n_train = int(TRAIN_FRAC * N)
n_val   = int(VAL_FRAC * N)
n_test  = N - n_train - n_val

idx_train = slice(0, n_train)
idx_val   = slice(n_train, n_train + n_val)
idx_test  = slice(n_train + n_val, N)

X_train, Y_train = X[idx_train], Y[idx_train]
X_val,   Y_val   = X[idx_val],   Y[idx_val]
X_test,  Y_test  = X[idx_test],  Y[idx_test]

print("Splits:")
print("  train:", X_train.shape)
print("  val  :", X_val.shape)
print("  test :", X_test.shape)

Splits:
  train: (24192, 384, 1)
  val  : (3456, 384, 1)
  test : (6912, 384, 1)


In [ ]:
# dataloading

class ElecSeq2SeqDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.from_numpy(X)  # (N, hist, D)
        self.Y = torch.from_numpy(Y)  # (N, hzn, D)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, i):
        return self.X[i], self.Y[i]

train_dl = DataLoader(
    ElecSeq2SeqDataset(X_train, Y_train),
    batch_size=BATCH, shuffle=True,
    num_workers=2, pin_memory=True
)
val_dl = DataLoader(
    ElecSeq2SeqDataset(X_val, Y_val),
    batch_size=BATCH, shuffle=False,
    num_workers=2, pin_memory=True
)
test_dl = DataLoader(
    ElecSeq2SeqDataset(X_test, Y_test),
    batch_size=BATCH, shuffle=False,
    num_workers=2, pin_memory=True
)

In [ ]:
# patching

def patchify(x, patch):
    # x: (B, T, D) -> (B, T//patch, D) by mean over non-overlap windows
    if patch == 1:
        return x
    B, T, D = x.shape
    T_trim = (T // patch) * patch
    x = x[:, :T_trim, :].reshape(B, T_trim // patch, patch, D).mean(dim=2)
    return x

def unpatchify(y_patch, patch, T_out):
    # y_patch: (B, Tp, D) -> (B, T_out, D) by repeat
    if patch == 1:
        return y_patch[:, :T_out, :]
    y = y_patch.repeat_interleave(patch, dim=1)
    return y[:, :T_out, :]

In [ ]:
# transformer

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=6000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        # x: (B, T, d_model)
        return x + self.pe[:, :x.size(1), :]

In [ ]:
class ElecSeq2SeqTransformer(nn.Module):
    def __init__(self,
                 d_feats,
                 d_model=256,
                 nhead=4,
                 num_layers=3,
                 dim_ff=512,
                 dropout=0.1,
                 hist_steps=HIST_STEPS,
                 hzn_steps=HZN_STEPS,
                 patch=PATCH):
        super().__init__()
        self.d_feats    = d_feats
        self.d_model    = d_model
        self.hist_steps = hist_steps
        self.hzn_steps  = hzn_steps
        self.patch      = patch

        self.in_proj = nn.Linear(d_feats, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.enc_pos = PositionalEncoding(d_model, max_len=(hist_steps // patch) + 16)

        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=num_layers)
        self.dec_pos = PositionalEncoding(d_model, max_len=(hzn_steps // patch) + 16)

        self.out_proj = nn.Linear(d_model, d_feats)

        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x):
        # x: (B, hist_steps, d_feats)
        B, T, D = x.shape
        x_p = patchify(x, self.patch)          # (B, T_p, D)
        z   = self.in_proj(x_p)                # (B, T_p, d_model)
        z   = self.enc_pos(z)
        mem = self.encoder(z)                  # (B, T_p, d_model)

        dec_Tp = self.hzn_steps // self.patch
        tgt = torch.zeros(B, dec_Tp, self.d_model, device=x.device)
        tgt = self.dec_pos(tgt)
        out = self.decoder(tgt, mem)           # (B, dec_Tp, d_model)
        y_p = self.out_proj(out)               # (B, dec_Tp, d_feats)
        y   = unpatchify(y_p, self.patch, self.hzn_steps)
        return y

In [ ]:
# losses

mse_loss = nn.MSELoss()

"""
def rmse_and_quantile(pred, true, q=0.5):
    mse = mse_loss(pred, true)
    rmse = torch.sqrt(mse + 1e-12)
    e = true - pred
    qloss = torch.mean(torch.maximum(q * e, (q - 1) * e))
    return rmse + qloss
"""

def rmse_only(pred, true):
    mse = mse_loss(pred, true)
    return torch.sqrt(mse + 1e-12)

def seq_rmse_np(y_true_np, y_pred_np):
    return float(np.sqrt(mean_squared_error(y_true_np.ravel(), y_pred_np.ravel())))

In [ ]:
# attacks

def fgsm_attack(attacker_model, X, Y, eps):
    X_adv = X.clone().detach().requires_grad_(True)
    pred  = attacker_model(X_adv)
    loss  = mse_loss(pred, Y)
    loss.backward()
    with torch.no_grad():
        X_pert = (X_adv + eps * X_adv.grad.sign()).clamp(0.0, 1.0)
    return X_pert.detach()

In [ ]:
def rfgsm_attack(attacker_model, X, Y, eps, alpha=None):
    if alpha is None:
        alpha = eps * RFGSM_ALPHA_FACTOR
    X0 = (X + torch.empty_like(X).uniform_(-eps, eps)).clamp(0.0, 1.0)
    X0 = X0.detach().requires_grad_(True)
    pred = attacker_model(X0)
    loss = mse_loss(pred, Y)
    loss.backward()
    with torch.no_grad():
        X_adv = (X0 + alpha * X0.grad.sign()).clamp(0.0, 1.0)
    return X_adv.detach()

In [ ]:
# training

def train_base(model, train_loader, val_loader,
               epochs=EPOCHS_BASE, lr=LR_BASE,
               ckpt_dir=BASE_MODEL_DIR):
    os.makedirs(ckpt_dir, exist_ok=True)
    opt = optim.AdamW(model.parameters(), lr=lr)
    best      = float("inf")
    best_path = os.path.join(ckpt_dir, "elec_base_best.pth")

    hb_flag, hb_thread = start_heartbeat("base_train")
    start_time = time.time()

    try:
        for ep in range(1, epochs + 1):
            model.train()
            tot = cnt = 0
            for xb, yb in train_loader:
                xb = xb.to(device)
                yb = yb.to(device)

                pred = model(xb)
                loss = rmse_only(pred, yb)

                opt.zero_grad()
                loss.backward()
                opt.step()

                bs = xb.size(0)
                tot += loss.item() * bs
                cnt += bs
            train_loss = tot / max(1, cnt)

            # Val
            model.eval()
            vt = vc = 0
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb = xb.to(device)
                    yb = yb.to(device)
                    pred = model(xb)
                    loss = rmse_only(pred, yb)
                    vt += loss.item() * xb.size(0)
                    vc += xb.size(0)
            val_loss = vt / max(1, vc)

            elapsed = (time.time() - start_time) / 60.0
            print(f"[Base] Epoch {ep:02d} | train {train_loss:.4f} | val {val_loss:.4f} | elapsed {elapsed:.2f} min")

            if val_loss < best:
                best = val_loss
                torch.save(model.state_dict(), best_path)
                print("  [Base] Saved best to", best_path, flush=True)

    finally:
        stop_heartbeat(hb_flag, hb_thread)

    return best_path

In [ ]:
@torch.no_grad()
def eval_one_epoch(model, loader):
    model.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        p = model(xb)
        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())
    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

In [ ]:
def eval_under_attack(model_to_eval, attacker_model, loader, atk_fn, atk_kwargs):
    model_to_eval.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        xa = atk_fn(attacker_model, xb, yb, **atk_kwargs)
        with torch.no_grad():
            p = model_to_eval(xa)
        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())
    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

In [ ]:
print("\ntraining base model")
base = ElecSeq2SeqTransformer(
    d_feats=D_FEATS,
    d_model=D_MODEL,
    nhead=N_HEAD,
    num_layers=N_LAYERS,
    dim_ff=DIM_FF,
    dropout=DROPOUT,
    hist_steps=HIST_STEPS,
    hzn_steps=HZN_STEPS,
    patch=PATCH
).to(device)

base_ckpt = train_base(base, train_dl, val_dl)
print("base checkpoint:", base_ckpt)


training base model


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


[Base] Epoch 01 | train 0.3347 | val 0.1897 | elapsed 0.57 min
  [Base] Saved best to /content/drive/MyDrive/298/Elec/SanityCheck/models/baseModel/elec_base_best.pth
[Base] Epoch 02 | train 0.1282 | val 0.1184 | elapsed 1.14 min
  [Base] Saved best to /content/drive/MyDrive/298/Elec/SanityCheck/models/baseModel/elec_base_best.pth
[Base] Epoch 03 | train 0.1211 | val 0.1079 | elapsed 1.70 min
  [Base] Saved best to /content/drive/MyDrive/298/Elec/SanityCheck/models/baseModel/elec_base_best.pth
[Base] Epoch 04 | train 0.1097 | val 0.1218 | elapsed 2.25 min
[Base] Epoch 05 | train 0.1076 | val 0.1130 | elapsed 2.81 min
[Base] Epoch 06 | train 0.1021 | val 0.1127 | elapsed 3.36 min
[Base] Epoch 07 | train 0.0983 | val 0.1139 | elapsed 3.91 min
[Base] Epoch 08 | train 0.0956 | val 0.1410 | elapsed 4.48 min
[Base] Epoch 09 | train 0.0925 | val 0.1146 | elapsed 5.05 min
[Base] Epoch 10 | train 0.0899 | val 0.1231 | elapsed 5.62 min
[Base] Epoch 11 | train 0.0882 | val 0.1289 | elapsed 6.18 mi

In [ ]:
# reload best base model on cpu for student generation
base_cpu = ElecSeq2SeqTransformer(
    d_feats=D_FEATS,
    d_model=D_MODEL,
    nhead=N_HEAD,
    num_layers=N_LAYERS,
    dim_ff=DIM_FF,
    dropout=DROPOUT,
    hist_steps=HIST_STEPS,
    hzn_steps=HZN_STEPS,
    patch=PATCH
).to("cpu")
base_cpu.load_state_dict(torch.load(base_ckpt, map_location="cpu"))

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


<All keys matched successfully>

In [ ]:
# student generation

num_students = 4
student_paths = []

print("\ncreating students from base")
for i in range(1, num_students + 1):
    st = copy.deepcopy(base_cpu)
    with torch.no_grad():
        for _, p in st.named_parameters():
            p.add_(1e-3 * torch.randn_like(p))
    outp = os.path.join(STUDENTS_DIR, f"student_{i:02d}.pth")
    torch.save(st.state_dict(), outp)
    student_paths.append(outp)
    print("saved student:", outp)

student_paths = sorted(student_paths)
print("students:", student_paths)


creating students from base
saved student: /content/drive/MyDrive/298/Elec/SanityCheck/fgsm/students/student_01.pth
saved student: /content/drive/MyDrive/298/Elec/SanityCheck/fgsm/students/student_02.pth
saved student: /content/drive/MyDrive/298/Elec/SanityCheck/fgsm/students/student_03.pth
saved student: /content/drive/MyDrive/298/Elec/SanityCheck/fgsm/students/student_04.pth
students: ['/content/drive/MyDrive/298/Elec/SanityCheck/fgsm/students/student_01.pth', '/content/drive/MyDrive/298/Elec/SanityCheck/fgsm/students/student_02.pth', '/content/drive/MyDrive/298/Elec/SanityCheck/fgsm/students/student_03.pth', '/content/drive/MyDrive/298/Elec/SanityCheck/fgsm/students/student_04.pth']


In [ ]:
# adv training of students

adv_student_paths = []

print("\nadversarial training of students")
for sp in student_paths:
    idx = os.path.basename(sp).split("_")[1].split(".")[0]
    print(f"\n[Student {idx}] adversarial training...")

    stu = ElecSeq2SeqTransformer(
        d_feats=D_FEATS,
        d_model=D_MODEL,
        nhead=N_HEAD,
        num_layers=N_LAYERS,
        dim_ff=DIM_FF,
        dropout=DROPOUT,
        hist_steps=HIST_STEPS,
        hzn_steps=HZN_STEPS,
        patch=PATCH
    ).to(device)
    stu.load_state_dict(torch.load(sp, map_location=device))

    opt = optim.AdamW(stu.parameters(), lr=LR_STUDENT)

    hb_flag, hb_thread = start_heartbeat(f"student_{idx}_rfgsm_train")
    start_time = time.time()

    try:
        for ep in range(1, EPOCHS_STU + 1):
            stu.train()
            tot = cnt = 0
            for xb, yb in train_dl:
                xb = xb.to(device)
                yb = yb.to(device)

                # clean prediction
                pred_c = stu(xb)
                Lc = rmse_only(pred_c, yb)

                # multi epsilon rfgsm adv training
                adv_losses = []
                for eps in FGSM_TRAIN_EPS_LIST:
                    xa = rfgsm_attack(stu, xb, yb, eps=eps)
                    pred_a = stu(xa)
                    La = rmse_only(pred_a, yb)
                    adv_losses.append(La)

                if len(adv_losses) > 0:
                    La_mean = torch.stack(adv_losses).mean()
                else:
                    La_mean = 0.0 * Lc

                loss = 0.25 * Lc + 0.75 * La_mean

                opt.zero_grad()
                loss.backward()
                opt.step()

                bs = xb.size(0)
                tot += loss.item() * bs
                cnt += bs

            avg_loss = tot / max(1, cnt)
            elapsed  = (time.time() - start_time) / 60.0
            print(f"  [Student {idx}] Epoch {ep:02d} | avg_loss={avg_loss:.4f} | elapsed={elapsed:.2f} min")

    finally:
        stop_heartbeat(hb_flag, hb_thread)

    outp = os.path.join(STUDENTS_ADV_DIR, f"student_{idx}_rfgsm_adv.pth")
    torch.save(stu.to("cpu").state_dict(), outp)
    adv_student_paths.append(outp)
    print(f"  [Student {idx}] Saved adv student:", outp)

adv_student_paths = sorted(adv_student_paths)


adversarial training of students

[Student 01] adversarial training...
  [Student 01] Epoch 01 | avg_loss=0.1295 | elapsed=4.41 min
  [Student 01] Epoch 02 | avg_loss=0.1129 | elapsed=8.84 min
[heartbeat:student_01_rfgsm_train] running... elapsed=10.00 min
  [Student 01] Epoch 03 | avg_loss=0.1087 | elapsed=13.27 min
  [Student 01] Epoch 04 | avg_loss=0.1060 | elapsed=17.69 min
[heartbeat:student_01_rfgsm_train] running... elapsed=20.00 min
  [Student 01] Epoch 05 | avg_loss=0.1038 | elapsed=22.11 min
  [Student 01] Saved adv student: /content/drive/MyDrive/298/Elec/SanityCheck/fgsm/students_adv/student_01_rfgsm_adv.pth

[Student 02] adversarial training...
  [Student 02] Epoch 01 | avg_loss=0.1325 | elapsed=4.42 min
  [Student 02] Epoch 02 | avg_loss=0.1139 | elapsed=8.85 min
[heartbeat:student_02_rfgsm_train] running... elapsed=10.00 min
  [Student 02] Epoch 03 | avg_loss=0.1093 | elapsed=13.27 min
  [Student 02] Epoch 04 | avg_loss=0.1068 | elapsed=17.69 min
[heartbeat:student_02_r

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, math, copy, random, time, threading, gc, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
DATA_PATH = "/content/drive/MyDrive/298/Elec/Data/LD2011_2014.txt"

SANITY_FGSM_DIR        = "/content/drive/MyDrive/298/Elec/SanityCheck/fgsm"
BASE_MODEL_DIR         = "/content/drive/MyDrive/298/Elec/SanityCheck/models/baseModel"
SANITY_RESULTS_DIR     = os.path.join(SANITY_FGSM_DIR, "results")
SANITY_ARTIFACTS_DIR   = os.path.join(SANITY_RESULTS_DIR, "artifacts")
SANITY_STUDENTS_ADV_DIR= os.path.join(SANITY_FGSM_DIR, "students_adv")

FINAL_PROJECT_DIR    = "/content/drive/MyDrive/298/Elec/FinalPipeline/fgsm"
FINAL_RESULTS_DIR    = os.path.join(FINAL_PROJECT_DIR, "results")
FINAL_PLOTS_DIR      = os.path.join(FINAL_RESULTS_DIR, "plots")
FINAL_ARTIFACTS_DIR  = os.path.join(FINAL_RESULTS_DIR, "artifacts")

for d in [FINAL_PROJECT_DIR, FINAL_RESULTS_DIR, FINAL_PLOTS_DIR, FINAL_ARTIFACTS_DIR]:
    os.makedirs(d, exist_ok=True)

FGSM_RUNS_CSV          = os.path.join(FINAL_RESULTS_DIR, "fgsm_morphence_runs.csv")
FGSM_STATS_CSV         = os.path.join(FINAL_RESULTS_DIR, "fgsm_morphence_stats.csv")
XFER_MATRIX_CSV        = os.path.join(FINAL_RESULTS_DIR, "fgsm_transfer_matrix.csv")
WEIGHT_DIVERSITY_CSV   = os.path.join(FINAL_RESULTS_DIR, "fgsm_weight_diversity.csv")
PRED_DIVERSITY_CSV     = os.path.join(FINAL_RESULTS_DIR, "fgsm_prediction_diversity.csv")

print("SanityCheck FGSM dir:", SANITY_FGSM_DIR)
print("SanityCheck artifacts dir:", SANITY_ARTIFACTS_DIR)
print("FinalPipeline FGSM results dir:", FINAL_RESULTS_DIR)

SanityCheck FGSM dir: /content/drive/MyDrive/298/Elec/SanityCheck/fgsm
SanityCheck artifacts dir: /content/drive/MyDrive/298/Elec/SanityCheck/fgsm/results/artifacts
FinalPipeline FGSM results dir: /content/drive/MyDrive/298/Elec/FinalPipeline/fgsm/results


In [ ]:
FREQ = "15min"
STEPS_PER_HOUR = 4
HOURS_PER_DAY  = 24

HIST_HOURS  = 96
HZN_HOURS   = 24
HIST_STEPS  = HIST_HOURS * STEPS_PER_HOUR   # 384
HZN_STEPS   = HZN_HOURS  * STEPS_PER_HOUR   # 96

BATCH       = 16
PATCH       = 4
D_FEATS     = 1
D_MODEL     = 256
N_HEAD      = 4
N_LAYERS    = 3
DIM_FF      = 512
DROPOUT     = 0.1

# FGSM configs
RFGSM_EVAL_EPS_LIST = [0.1, 0.2, 0.3, 0.5]
RFGSM_ALPHA_FACTOR  = 1.0

N_RUNS = 30
HEARTBEAT_SECS = 600  # 10 minutes

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", device)

Device: cuda


In [ ]:
#utilities

def start_heartbeat(name="task"):
    flag = {"on": True}
    start_time = time.time()

    def _hb():
        while flag["on"]:
            time.sleep(HEARTBEAT_SECS)
            if flag["on"]:
                elapsed = (time.time() - start_time) / 60.0
                print(f"[heartbeat:{name}] running... elapsed={elapsed:.2f} min", flush=True)

    t = threading.Thread(target=_hb, daemon=True)
    t.start()
    return flag, t

def stop_heartbeat(flag, thread):
    flag["on"] = False
    try:
        thread.join(timeout=1)
    except Exception:
        pass

def seq_rmse_np(y_true_np, y_pred_np):
    return float(np.sqrt(mean_squared_error(y_true_np.ravel(), y_pred_np.ravel())))

In [ ]:
#load test set

X_test_path = os.path.join(SANITY_ARTIFACTS_DIR, "X_test.npy")
Y_test_path = os.path.join(SANITY_ARTIFACTS_DIR, "Y_test.npy")

assert os.path.exists(X_test_path), f"Missing {X_test_path}"
assert os.path.exists(Y_test_path), f"Missing {Y_test_path}"

X_test = np.load(X_test_path)
Y_test = np.load(Y_test_path)

print("Loaded X_test:", X_test.shape)
print("Loaded Y_test:", Y_test.shape)

class ElecSeq2SeqDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.from_numpy(X)  # (N, hist, D)
        self.Y = torch.from_numpy(Y)  # (N, hzn, D)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, i):
        return self.X[i], self.Y[i]

test_dl = DataLoader(
    ElecSeq2SeqDataset(X_test, Y_test),
    batch_size=BATCH,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

Loaded X_test: (6912, 384, 1)
Loaded Y_test: (6912, 96, 1)


In [ ]:
#model definition

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=6000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


In [ ]:
#patching

def patchify(x, patch):
    if patch == 1:
        return x
    B, T, D = x.shape
    T_trim = (T // patch) * patch
    x = x[:, :T_trim, :].reshape(B, T_trim // patch, patch, D).mean(dim=2)
    return x

def unpatchify(y_patch, patch, T_out):
    if patch == 1:
        return y_patch[:, :T_out, :]
    y = y_patch.repeat_interleave(patch, dim=1)
    return y[:, :T_out, :]

In [ ]:
#transformer

class ElecSeq2SeqTransformer(nn.Module):
    def __init__(self,
                 d_feats,
                 d_model=256,
                 nhead=4,
                 num_layers=3,
                 dim_ff=512,
                 dropout=0.1,
                 hist_steps=HIST_STEPS,
                 hzn_steps=HZN_STEPS,
                 patch=PATCH):
        super().__init__()
        self.d_feats    = d_feats
        self.d_model    = d_model
        self.hist_steps = hist_steps
        self.hzn_steps  = hzn_steps
        self.patch      = patch

        self.in_proj = nn.Linear(d_feats, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.enc_pos = PositionalEncoding(d_model, max_len=(hist_steps // patch) + 16)

        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=num_layers)
        self.dec_pos = PositionalEncoding(d_model, max_len=(hzn_steps // patch) + 16)

        self.out_proj = nn.Linear(d_model, d_feats)

        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x):
        B, T, D = x.shape
        x_p = patchify(x, self.patch)
        z   = self.in_proj(x_p)
        z   = self.enc_pos(z)
        mem = self.encoder(z)

        dec_Tp = self.hzn_steps // self.patch
        tgt = torch.zeros(B, dec_Tp, self.d_model, device=x.device)
        tgt = self.dec_pos(tgt)
        out = self.decoder(tgt, mem)
        y_p = self.out_proj(out)
        y   = unpatchify(y_p, self.patch, self.hzn_steps)
        return y

In [ ]:
#fgsm attack

mse_loss = nn.MSELoss()

def rfgsm_attack(attacker_model, X, Y, eps, alpha=None, return_grad_norm=False):

    #random start FGSM with antimasking safeguards.

    attacker_model.eval()

    if alpha is None:
        alpha = eps * RFGSM_ALPHA_FACTOR

    noise = torch.empty_like(X).uniform_(-eps, eps)
    X0 = (X + noise).clamp(0.0, 1.0)
    X0 = X0.detach().requires_grad_(True)

    pred = attacker_model(X0)
    loss = mse_loss(pred, Y)

    grad = torch.autograd.grad(loss, X0, only_inputs=True, retain_graph=False)[0]
    grad_sign = grad.sign()

    X_adv = (X0 + alpha * grad_sign).clamp(0.0, 1.0)

    if return_grad_norm:
        gn = grad.view(grad.size(0), -1).norm(p=2, dim=1)
        return X_adv.detach(), gn.detach()
    return X_adv.detach()

In [ ]:
#load base

base_ckpt = os.path.join(BASE_MODEL_DIR, "elec_base_best.pth")
assert os.path.exists(base_ckpt), f"Missing base checkpoint: {base_ckpt}"

best_base = ElecSeq2SeqTransformer(
    d_feats=D_FEATS,
    d_model=D_MODEL,
    nhead=N_HEAD,
    num_layers=N_LAYERS,
    dim_ff=DIM_FF,
    dropout=DROPOUT,
    hist_steps=HIST_STEPS,
    hzn_steps=HZN_STEPS,
    patch=PATCH
).to(device)
best_base.load_state_dict(torch.load(base_ckpt, map_location=device))
best_base.eval()
print("Loaded base model from:", base_ckpt)


#load students

adv_student_paths = sorted(
    [
        os.path.join(SANITY_STUDENTS_ADV_DIR, f)
        for f in os.listdir(SANITY_STUDENTS_ADV_DIR)
        if f.endswith("_rfgsm_adv.pth")
    ]
)

assert len(adv_student_paths) > 0, "No *_rfgsm_adv.pth found in students_adv dir!"
print("Adv students (from SanityCheck):")
for p in adv_student_paths:
    print("  ", p)

adv_students = []
for sp in adv_student_paths:
    stu = ElecSeq2SeqTransformer(
        d_feats=D_FEATS,
        d_model=D_MODEL,
        nhead=N_HEAD,
        num_layers=N_LAYERS,
        dim_ff=DIM_FF,
        dropout=DROPOUT,
        hist_steps=HIST_STEPS,
        hzn_steps=HZN_STEPS,
        patch=PATCH
    ).to(device)
    stu.load_state_dict(torch.load(sp, map_location=device))
    stu.eval()
    adv_students.append(stu)

student_names = [os.path.basename(p).replace(".pth", "") for p in adv_student_paths]
print(f"Loaded {len(adv_students)} adversarially trained students.")

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Loaded base model from: /content/drive/MyDrive/298/Elec/SanityCheck/models/baseModel/elec_base_best.pth
Adv students (from SanityCheck):
   /content/drive/MyDrive/298/Elec/SanityCheck/fgsm/students_adv/student_01_rfgsm_adv.pth
   /content/drive/MyDrive/298/Elec/SanityCheck/fgsm/students_adv/student_02_rfgsm_adv.pth
   /content/drive/MyDrive/298/Elec/SanityCheck/fgsm/students_adv/student_03_rfgsm_adv.pth
   /content/drive/MyDrive/298/Elec/SanityCheck/fgsm/students_adv/student_04_rfgsm_adv.pth
Loaded 4 adversarially trained students.


In [ ]:
#eval helpers

@torch.no_grad()
def eval_clean_rmse(model, loader):
    model.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        p  = model(xb)
        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())
    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

@torch.no_grad()
def eval_ensemble_rmse(models, loader):
    for m in models:
        m.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        preds = []
        for m in models:
            preds.append(m(xb))
        preds = torch.stack(preds, dim=0).mean(dim=0)  # (B,H,D)
        ys.append(yb.cpu().numpy())
        ps.append(preds.cpu().numpy())
    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

def eval_pair_rfgsm(defender, attacker, loader, eps, alpha=None):
    defender.eval()
    attacker.eval()
    if alpha is None:
        alpha = eps * RFGSM_ALPHA_FACTOR

    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        xa = rfgsm_attack(attacker, xb, yb, eps=eps, alpha=alpha)
        with torch.no_grad():
            p = defender(xa)
        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())
    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

def eval_random_switch_rfgsm(models, loader, eps, alpha=None):
    if alpha is None:
        alpha = eps * RFGSM_ALPHA_FACTOR

    for m in models:
        m.eval()

    ys, ps = [], []
    n_stu = len(models)

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        atk_idx  = random.randrange(n_stu)
        def_idx  = random.randrange(n_stu)
        attacker = models[atk_idx]
        defender = models[def_idx]

        xa = rfgsm_attack(attacker, xb, yb, eps=eps, alpha=alpha)
        with torch.no_grad():
            p = defender(xa)
        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())

    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

In [ ]:
#diversity metrics

def flatten_params(model):
    return torch.cat([p.detach().cpu().view(-1) for p in model.parameters()])

def compute_weight_diversity(students, names):
    vecs = [flatten_params(m) for m in students]
    k = len(vecs)
    rows = []
    for i, j in itertools.combinations(range(k), 2):
        d = torch.norm(vecs[i] - vecs[j], p=2).item()
        rows.append({
            "student_i": names[i],
            "student_j": names[j],
            "l2_distance": d,
        })
    df_w = pd.DataFrame(rows)
    df_w.to_csv(WEIGHT_DIVERSITY_CSV, index=False)
    print("Saved weight diversity to:", WEIGHT_DIVERSITY_CSV)
    return df_w

@torch.no_grad()
def compute_prediction_diversity(students, loader):
    for m in students:
        m.eval()

    k = len(students)
    H = HZN_STEPS

    sum_pred  = np.zeros(H, dtype=np.float64)
    sum_pred2 = np.zeros(H, dtype=np.float64)
    count     = 0

    for xb, yb in loader:
        xb = xb.to(device)
        preds_stu = []
        for m in students:
            preds_stu.append(m(xb).cpu().numpy())  # (B,H,1)
        preds_stu = np.stack(preds_stu, axis=0)  # (k,B,H,1)

        preds_mean_batch = preds_stu.mean(axis=1).squeeze(-1)  # (k,H)

        sum_pred  += preds_mean_batch.mean(axis=0)
        sum_pred2 += (preds_mean_batch ** 2).mean(axis=0)
        count     += 1

    mean_pred  = sum_pred / max(1, count)
    mean_pred2 = sum_pred2 / max(1, count)
    var_pred   = np.maximum(mean_pred2 - mean_pred**2, 0.0)

    df_p = pd.DataFrame({
        "horizon_idx": np.arange(H),
        "mean_pred": mean_pred,
        "var_pred": var_pred,
    })
    df_p.to_csv(PRED_DIVERSITY_CSV, index=False)
    print("Saved prediction diversity to:", PRED_DIVERSITY_CSV)
    return df_p

print("\nComputing diversity metrics (once)...")
_ = compute_weight_diversity(adv_students, student_names)
_ = compute_prediction_diversity(adv_students, test_dl)


Computing diversity metrics (once)...
Saved weight diversity to: /content/drive/MyDrive/298/Elec/FinalPipeline/fgsm/results/fgsm_weight_diversity.csv
Saved prediction diversity to: /content/drive/MyDrive/298/Elec/FinalPipeline/fgsm/results/fgsm_prediction_diversity.csv


In [ ]:
#transferability matrix

xfer_rows = []
for eps in RFGSM_EVAL_EPS_LIST:
    alpha = eps * RFGSM_ALPHA_FACTOR
    print(f"\nComputing transfer matrix for eps={eps}...")
    for i, atk_name in enumerate(student_names):
        for j, def_name in enumerate(student_names):
            atk_model = adv_students[i]
            def_model = adv_students[j]
            rmse_ij = eval_pair_rfgsm(def_model, atk_model, test_dl, eps=eps, alpha=alpha)
            xfer_rows.append({
                "epsilon": eps,
                "attacker": atk_name,
                "defender": def_name,
                "rmse_adv": rmse_ij,
            })
            print(f"  atk={atk_name} -> def={def_name} | eps={eps:.2f} | RMSE={rmse_ij:.5f}")

df_xfer = pd.DataFrame(xfer_rows)
df_xfer.to_csv(XFER_MATRIX_CSV, index=False)
print("\nSaved transfer matrix to:", XFER_MATRIX_CSV)


Computing transfer matrix for eps=0.1...
  atk=student_01_rfgsm_adv -> def=student_01_rfgsm_adv | eps=0.10 | RMSE=0.14790
  atk=student_01_rfgsm_adv -> def=student_02_rfgsm_adv | eps=0.10 | RMSE=0.13826
  atk=student_01_rfgsm_adv -> def=student_03_rfgsm_adv | eps=0.10 | RMSE=0.13588
  atk=student_01_rfgsm_adv -> def=student_04_rfgsm_adv | eps=0.10 | RMSE=0.14379
  atk=student_02_rfgsm_adv -> def=student_01_rfgsm_adv | eps=0.10 | RMSE=0.14104
  atk=student_02_rfgsm_adv -> def=student_02_rfgsm_adv | eps=0.10 | RMSE=0.14323
  atk=student_02_rfgsm_adv -> def=student_03_rfgsm_adv | eps=0.10 | RMSE=0.13623
  atk=student_02_rfgsm_adv -> def=student_04_rfgsm_adv | eps=0.10 | RMSE=0.14271
  atk=student_03_rfgsm_adv -> def=student_01_rfgsm_adv | eps=0.10 | RMSE=0.14084
  atk=student_03_rfgsm_adv -> def=student_02_rfgsm_adv | eps=0.10 | RMSE=0.13836
  atk=student_03_rfgsm_adv -> def=student_03_rfgsm_adv | eps=0.10 | RMSE=0.14002
  atk=student_03_rfgsm_adv -> def=student_04_rfgsm_adv | eps=0.10 |

In [ ]:

#single run eval
#one fixed seed
#base vs adv student ensemble


FGSM_SINGLE_RUN_CSV = os.path.join(FINAL_RESULTS_DIR, "fgsm_morphence_single_run.csv")

print("\nSingle run eval: base vs ensemble")
print("Epsilons:", RFGSM_EVAL_EPS_LIST)

#fix single seed
single_seed = 1234
random.seed(single_seed)
np.random.seed(single_seed)
torch.manual_seed(single_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(single_seed)

single_rows = []   # ["attack", "epsilon", "model", "RMSE"]

#clean performance
#base vs ensemble of adv students
base_clean_single   = eval_clean_rmse(best_base,  test_dl)
morph_clean_single  = eval_ensemble_rmse(adv_students, test_dl)

single_rows.append(["clean", None, "base",           base_clean_single])
single_rows.append(["clean", None, "morph_ensemble", morph_clean_single])

print(
    f"[Single-run] clean | base={base_clean_single:.5f} | "
    f"morph_ensemble={morph_clean_single:.5f}",
    flush=True
)

#FGSM robustness
#base white box vs random switching
for eps in RFGSM_EVAL_EPS_LIST:
    alpha = eps * RFGSM_ALPHA_FACTOR

    #base attacked by itself
    base_adv_single = eval_pair_rfgsm(
        defender=best_base,
        attacker=best_base,
        loader=test_dl,
        eps=eps,
        alpha=alpha
    )

    #random attacker/defender switching among adv students
    morph_rand_single = eval_random_switch_rfgsm(
        models=adv_students,
        loader=test_dl,
        eps=eps,
        alpha=alpha
    )

    single_rows.append(["rfgsm", eps, "base",        base_adv_single])
    single_rows.append(["rfgsm", eps, "morph_rand",  morph_rand_single])

    print(
        f"[Single run] eps={eps:.2f} | base_adv={base_adv_single:.5f} | "
        f"morph_rand={morph_rand_single:.5f}",
        flush=True
    )

#save single run results
df_single = pd.DataFrame(
    single_rows,
    columns=["attack", "epsilon", "model", "RMSE"]
)
df_single.to_csv(FGSM_SINGLE_RUN_CSV, index=False)
print("\nSaved single run eval CSV to:", FGSM_SINGLE_RUN_CSV)




Single run eval: base vs ensemble
Epsilons: [0.1, 0.2, 0.3, 0.5]
[Single-run] clean | base=0.11506 | morph_ensemble=0.11763
[Single run] eps=0.10 | base_adv=0.16904 | morph_rand=0.14105
[Single run] eps=0.20 | base_adv=0.23030 | morph_rand=0.13930
[Single run] eps=0.30 | base_adv=0.27071 | morph_rand=0.14550
[Single run] eps=0.50 | base_adv=0.29807 | morph_rand=0.15721

Saved single run eval CSV to: /content/drive/MyDrive/298/Elec/FinalPipeline/fgsm/results/fgsm_morphence_single_run.csv


In [ ]:
#30 run eval

print("\n30 run FGSM eval: base vs ensemble")
print("Epsilons:", RFGSM_EVAL_EPS_LIST)
print("Runs:", N_RUNS)

all_rows = []   # ["attack", "epsilon", "model", "RMSE", "run"]

start_all = time.time()
hb_flag, hb_thread = start_heartbeat("eval_fgsm_morphence")

try:
    for run_i in range(1, N_RUNS + 1):
        run_start = time.time()
        seed_i = 3000 + run_i
        random.seed(seed_i)
        np.random.seed(seed_i)
        torch.manual_seed(seed_i)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed_i)

        #clean
        base_clean  = eval_clean_rmse(best_base, test_dl)
        morph_clean = eval_ensemble_rmse(adv_students, test_dl)

        all_rows.append(["clean", None, "base",           base_clean,  run_i])
        all_rows.append(["clean", None, "morph_ensemble", morph_clean, run_i])

        print(f"[Run {run_i:02d}] clean | base={base_clean:.5f} | morph_ens={morph_clean:.5f}",
              flush=True)

        #FGSM (white box for base, random switch for ensemble)
        for eps in RFGSM_EVAL_EPS_LIST:
            alpha = eps * RFGSM_ALPHA_FACTOR

            base_adv   = eval_pair_rfgsm(best_base, best_base, test_dl, eps=eps, alpha=alpha)
            morph_rand = eval_random_switch_rfgsm(adv_students, test_dl, eps=eps, alpha=alpha)

            all_rows.append(["rfgsm", eps, "base",       base_adv,   run_i])
            all_rows.append(["rfgsm", eps, "morph_rand", morph_rand, run_i])

            print(
                f"[Run {run_i:02d}] eps={eps:.2f} | base_adv={base_adv:.5f} | "
                f"morph_rand={morph_rand:.5f}",
                flush=True
            )

        # autosave after each run
        df_runs = pd.DataFrame(
            all_rows,
            columns=["attack", "epsilon", "model", "RMSE", "run"]
        )
        df_runs.to_csv(FGSM_RUNS_CSV, index=False)

        stats = (
            df_runs
            .groupby(["attack", "epsilon", "model"])["RMSE"]
            .agg(mean="mean", std="std", n="count")
            .reset_index()
            .sort_values(["attack", "epsilon", "model"])
        )
        stats.to_csv(FGSM_STATS_CSV, index=False)

        run_mins   = (time.time() - run_start) / 60.0
        total_mins = (time.time() - start_all) / 60.0
        print(
            f"Completed run {run_i}/{N_RUNS} | "
            f"run_time={run_mins:.2f} min | total_elapsed={total_mins:.2f} min",
            flush=True
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        gc.collect()

finally:
    stop_heartbeat(hb_flag, hb_thread)

print("\nSaved FGSM ensemble eval runs CSV to:", FGSM_RUNS_CSV)
print("Saved FGSM ensemble eval stats CSV to:", FGSM_STATS_CSV)
print("Weight diversity CSV:", WEIGHT_DIVERSITY_CSV)
print("Prediction diversity CSV:", PRED_DIVERSITY_CSV)
print("Transfer matrix CSV:", XFER_MATRIX_CSV)
print("FinalPipeline plots dir:", FINAL_PLOTS_DIR)
print("FinalPipeline artifacts dir:", FINAL_ARTIFACTS_DIR)


30 run FGSM eval: base vs ensemble
Epsilons: [0.1, 0.2, 0.3, 0.5]
Runs: 30
[Run 01] clean | base=0.11506 | morph_ens=0.11763
[Run 01] eps=0.10 | base_adv=0.16974 | morph_rand=0.14203
[Run 01] eps=0.20 | base_adv=0.22918 | morph_rand=0.13836
[Run 01] eps=0.30 | base_adv=0.27167 | morph_rand=0.14516
[Run 01] eps=0.50 | base_adv=0.29763 | morph_rand=0.15922
Completed run 1/30 | run_time=1.34 min | total_elapsed=1.34 min
[Run 02] clean | base=0.11506 | morph_ens=0.11763
[Run 02] eps=0.10 | base_adv=0.16861 | morph_rand=0.14126
[Run 02] eps=0.20 | base_adv=0.22999 | morph_rand=0.13928
[Run 02] eps=0.30 | base_adv=0.27259 | morph_rand=0.14621
[Run 02] eps=0.50 | base_adv=0.29815 | morph_rand=0.15863
Completed run 2/30 | run_time=1.33 min | total_elapsed=2.66 min
[Run 03] clean | base=0.11506 | morph_ens=0.11763
[Run 03] eps=0.10 | base_adv=0.16907 | morph_rand=0.14040
[Run 03] eps=0.20 | base_adv=0.22973 | morph_rand=0.13785
[Run 03] eps=0.30 | base_adv=0.27178 | morph_rand=0.14646
[Run 03]

In [ ]:
import os
import pandas as pd
import numpy as np

RESULTS_DIR = FINAL_RESULTS_DIR
RUNS_CSV    = FGSM_RUNS_CSV
FIG_DIR     = RESULTS_DIR
os.makedirs(FIG_DIR, exist_ok=True)

df_runs = pd.read_csv(RUNS_CSV)
print("Loaded runs:", df_runs.shape)
print(df_runs.head())


# Clean stats (base vs morph_ensemble)

df_clean = df_runs[df_runs["attack"] == "clean"].copy()

# ensure nice ordering of models
df_clean["model"] = pd.Categorical(
    df_clean["model"],
    categories=["base", "morph_ensemble"],
    ordered=True
)

clean_stats = (
    df_clean
    .groupby("model")["RMSE"]
    .agg(
        mean="mean",
        std="std",
        min="min",
        q1=lambda s: s.quantile(0.25),
        median="median",
        q3=lambda s: s.quantile(0.75),
        max="max",
        n="count",
    )
    .reset_index()
    .sort_values("model")
)

print("\n[Clean] stats:")
print(clean_stats.to_string(index=False))

clean_csv = os.path.join(FIG_DIR, "fgsm_clean_stats.csv")
clean_stats.to_csv(clean_csv, index=False)
print("Saved clean stats to:", clean_csv)


# FGSM stats (base vs morph_rand)

df_rfgsm = df_runs[df_runs["attack"] == "rfgsm"].copy()

def model_rfgsm_stats(df, model_name):
    st = (
        df[df["model"] == model_name]
        .groupby("epsilon")["RMSE"]
        .agg(
            mean="mean",
            std="std",
            min="min",
            q1=lambda s: s.quantile(0.25),
            median="median",
            q3=lambda s: s.quantile(0.75),
            max="max",
            n="count",
        )
        .reset_index()
        .sort_values("epsilon")
    )

    print(f"\n[rFGSM] {model_name} stats:")
    print(st.to_string(index=False))

    out_csv = os.path.join(FIG_DIR, f"fgsm_rfgsm_{model_name}_stats.csv")
    st.to_csv(out_csv, index=False)
    print(f"Saved: {out_csv}")

# base white-box stats
model_rfgsm_stats(df_rfgsm, "base")

# Morphence: random-switch ensemble under rFGSM
model_rfgsm_stats(df_rfgsm, "morph_rand")


Loaded runs: (300, 5)
  attack  epsilon           model      RMSE  run
0  clean      NaN            base  0.115062    1
1  clean      NaN  morph_ensemble  0.117630    1
2  rfgsm      0.1            base  0.169737    1
3  rfgsm      0.1      morph_rand  0.142028    1
4  rfgsm      0.2            base  0.229181    1

[Clean] stats:
         model     mean  std      min       q1   median       q3      max  n
          base 0.115062  0.0 0.115062 0.115062 0.115062 0.115062 0.115062 30
morph_ensemble 0.117630  0.0 0.117630 0.117630 0.117630 0.117630 0.117630 30
Saved clean stats to: /content/drive/MyDrive/298/Elec/FinalPipeline/fgsm/results/fgsm_clean_stats.csv

[rFGSM] base stats:
 epsilon     mean      std      min       q1   median       q3      max  n
     0.1 0.169164 0.000460 0.168365 0.168848 0.169102 0.169434 0.170091 30
     0.2 0.230447 0.000853 0.228434 0.229911 0.230401 0.230958 0.232460 30
     0.3 0.272065 0.000679 0.270698 0.271689 0.272034 0.272527 0.273530 30
     0.5 0.298

/tmp/ipython-input-83165987.py:28: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("model")["RMSE"]


<h1>BIM</h1>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, math, copy, random, time, threading, gc, traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


In [ ]:
DATA_PATH = "/content/drive/MyDrive/298/Elec/Data/LD2011_2014.txt"

BASE_MODEL_DIR = "/content/drive/MyDrive/298/Elec/SanityCheck/models/baseModel"
BASE_CKPT      = os.path.join(BASE_MODEL_DIR, "elec_base_best.pth")

PROJECT_DIR    = "/content/drive/MyDrive/298/Elec/SanityCheck/bim"

STUDENTS_DIR     = os.path.join(PROJECT_DIR, "students")
STUDENTS_ADV_DIR = os.path.join(PROJECT_DIR, "students_adv")
RESULTS_DIR      = os.path.join(PROJECT_DIR, "results")
PLOTS_DIR        = os.path.join(RESULTS_DIR, "plots")
ARTIFACTS_DIR    = os.path.join(RESULTS_DIR, "artifacts")
CRASH_DIR        = os.path.join(PROJECT_DIR, "crash_dump")

for d in [PROJECT_DIR, STUDENTS_DIR, STUDENTS_ADV_DIR, RESULTS_DIR, PLOTS_DIR, ARTIFACTS_DIR, CRASH_DIR]:
    os.makedirs(d, exist_ok=True)

RUNS_CSV          = os.path.join(RESULTS_DIR, "bim_runs.csv")
STATS_CSV         = os.path.join(RESULTS_DIR, "bim_stats.csv")
PER_HORIZON_CSV   = os.path.join(RESULTS_DIR, "bim_per_horizon_rmse.csv")
GRAD_NORMS_CSV    = os.path.join(RESULTS_DIR, "bim_grad_norms.csv")
MODEL_SIZES_CSV   = os.path.join(RESULTS_DIR, "bim_model_sizes.csv")
ARTIFACT_INDEX_CSV= os.path.join(RESULTS_DIR, "bim_artifact_index.csv")
ERROR_LOG         = os.path.join(CRASH_DIR, "error_log.txt")

In [ ]:
FREQ = "15min"
STEPS_PER_HOUR = 4
HOURS_PER_DAY  = 24

HIST_HOURS  = 96
HZN_HOURS   = 24
HIST_STEPS  = HIST_HOURS * STEPS_PER_HOUR   # 384
HZN_STEPS   = HZN_HOURS  * STEPS_PER_HOUR   # 96
SLIDE       = 1

TRAIN_FRAC  = 0.70
VAL_FRAC    = 0.10

BATCH       = 16
PATCH       = 4
D_FEATS     = 1
D_MODEL     = 256
N_HEAD      = 4
N_LAYERS    = 3
DIM_FF      = 512
DROPOUT     = 0.1

LR_STUDENT  = 1e-4
EPOCHS_STU  = 5

In [ ]:
BIM_TRAIN_EPS_LIST = [0.1, 0.2, 0.3, 0.5]
BIM_EVAL_EPS_LIST  = [0.1, 0.2, 0.3, 0.5]
BIM_ITERS          = 10


# random start choices:
BIM_RANDOM_START_TRAIN = False
BIM_RANDOM_START_EVAL  = True

N_RUNS = 30

HEARTBEAT_SECS = 600  # 10 minutes

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", device)

Device: cpu


In [ ]:
def start_heartbeat(name="task"):
    flag = {"on": True}
    start_time = time.time()

    def _hb():
        while flag["on"]:
            time.sleep(HEARTBEAT_SECS)
            if flag["on"]:
                elapsed = (time.time() - start_time) / 60.0
                print(f"[heartbeat:{name}] running... elapsed={elapsed:.2f} min", flush=True)

    t = threading.Thread(target=_hb, daemon=True)
    t.start()
    return flag, t

def stop_heartbeat(flag, thread):
    flag["on"] = False
    try:
        thread.join(timeout=1)
    except Exception:
        pass

def append_df_to_csv(df, path):
    header = not os.path.exists(path)
    df.to_csv(path, mode="a", header=header, index=False)

def model_n_params(model: nn.Module):
    return sum(p.numel() for p in model.parameters())

def model_file_size_mb(path: str):
    if not os.path.exists(path):
        return None
    return os.path.getsize(path) / (1024.0 * 1024.0)

def log_error(msg: str):
    with open(ERROR_LOG, "a") as f:
        f.write(msg + "\n")

In [ ]:
print("Reading raw txt:", DATA_PATH)
df = pd.read_csv(
    DATA_PATH,
    sep=';',
    quotechar='"',
    decimal=',',
    parse_dates=[0],
    index_col=0,
    na_values=['', 'NA', 'NaN'],
    low_memory=False
)
df.index.name = "timestamp"
df = df.sort_index()
print("Raw shape:", df.shape)

METER = "MT_001"
print("Using single meter:", METER)

# restrict to 2013 only
START_DT = pd.Timestamp("2013-01-01 00:00:00")
END_DT   = pd.Timestamp("2013-12-31 23:45:00")

df = df[[METER]]
df = df.loc[(df.index >= START_DT) & (df.index <= END_DT)].copy()

# regularize index
full_idx = pd.date_range(START_DT, END_DT, freq=FREQ)
df = df.reindex(full_idx)

# fill up to 4 consecutive missing steps forward & backward
df = df.apply(pd.to_numeric, errors='coerce')
df = df.ffill(limit=4).bfill(limit=4)
print("Post filtering shape:", df.shape)

# scale on early TRAIN_FRAC portion of time
n_t       = len(df)
split_idx = int(TRAIN_FRAC * n_t)
df_train  = df.iloc[:split_idx]
df_all    = df

scaler = MinMaxScaler()
train_scaled = pd.DataFrame(
    scaler.fit_transform(df_train),
    index=df_train.index,
    columns=df_train.columns
)
full_scaled = pd.DataFrame(
    scaler.transform(df_all),
    index=df_all.index,
    columns=df_all.columns
)
print("Scaled shape:", full_scaled.shape)

Reading raw txt: /content/drive/MyDrive/298/Elec/Data/LD2011_2014.txt
Raw shape: (140256, 370)
Using single meter: MT_001
Post filtering shape: (35040, 1)
Scaled shape: (35040, 1)


In [ ]:
values = full_scaled.values.astype(np.float32)  # (T, 1)
T, D = values.shape
assert D == 1
print("T, D:", T, D)

X_list, Y_list, meta = [], [], []
for t in range(HIST_STEPS, T - HZN_STEPS, SLIDE):
    x = values[t - HIST_STEPS:t, :]      # (384, 1)
    y = values[t:t + HZN_STEPS, :]       # (96, 1)
    X_list.append(x)
    Y_list.append(y)
    meta.append((
        full_scaled.index[t - HIST_STEPS],
        full_scaled.index[t - 1],
        full_scaled.index[t],
        full_scaled.index[t + HZN_STEPS - 1],
    ))

X = np.stack(X_list)   # (N, 384, 1)
Y = np.stack(Y_list)   # (N, 96, 1)
meta_df = pd.DataFrame(meta, columns=["x_start", "x_end", "y_start", "y_end"])
print("Windowed tensors:", X.shape, Y.shape)

T, D: 35040 1
Windowed tensors: (34560, 384, 1) (34560, 96, 1)


In [ ]:
N = len(X)
n_train = int(TRAIN_FRAC * N)
n_val   = int(VAL_FRAC * N)
n_test  = N - n_train - n_val

idx_train = slice(0, n_train)
idx_val   = slice(n_train, n_train + n_val)
idx_test  = slice(n_train + n_val, N)

X_train, Y_train = X[idx_train], Y[idx_train]
X_val,   Y_val   = X[idx_val],   Y[idx_val]
X_test,  Y_test  = X[idx_test],  Y[idx_test]

print("Splits:")
print("  train:", X_train.shape)
print("  val  :", X_val.shape)
print("  test :", X_test.shape)

np.save(os.path.join(ARTIFACTS_DIR, "X_test.npy"), X_test)
np.save(os.path.join(ARTIFACTS_DIR, "Y_test.npy"), Y_test)
print("Saved full test X/Y arrays to ARTIFACTS_DIR.")

Splits:
  train: (24192, 384, 1)
  val  : (3456, 384, 1)
  test : (6912, 384, 1)
Saved full test X/Y arrays to ARTIFACTS_DIR.


In [ ]:
class ElecSeq2SeqDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.from_numpy(X)  # (N, hist, D)
        self.Y = torch.from_numpy(Y)  # (N, hzn, D)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, i):
        return self.X[i], self.Y[i]

train_dl = DataLoader(
    ElecSeq2SeqDataset(X_train, Y_train),
    batch_size=BATCH, shuffle=True,
    num_workers=2, pin_memory=True
)
val_dl = DataLoader(
    ElecSeq2SeqDataset(X_val, Y_val),
    batch_size=BATCH, shuffle=False,
    num_workers=2, pin_memory=True
)
test_dl = DataLoader(
    ElecSeq2SeqDataset(X_test, Y_test),
    batch_size=BATCH, shuffle=False,
    num_workers=2, pin_memory=True
)

In [ ]:
def patchify(x, patch):
    if patch == 1:
        return x
    B, T, D = x.shape
    T_trim = (T // patch) * patch
    x = x[:, :T_trim, :].reshape(B, T_trim // patch, patch, D).mean(dim=2)
    return x

def unpatchify(y_patch, patch, T_out):
    if patch == 1:
        return y_patch[:, :T_out, :]
    y = y_patch.repeat_interleave(patch, dim=1)
    return y[:, :T_out, :]

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=6000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class ElecSeq2SeqTransformer(nn.Module):
    def __init__(self,
                 d_feats,
                 d_model=256,
                 nhead=4,
                 num_layers=3,
                 dim_ff=512,
                 dropout=0.1,
                 hist_steps=HIST_STEPS,
                 hzn_steps=HZN_STEPS,
                 patch=PATCH):
        super().__init__()
        self.d_feats    = d_feats
        self.d_model    = d_model
        self.hist_steps = hist_steps
        self.hzn_steps  = hzn_steps
        self.patch      = patch

        self.in_proj = nn.Linear(d_feats, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.enc_pos = PositionalEncoding(d_model, max_len=(hist_steps // patch) + 16)

        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=num_layers)
        self.dec_pos = PositionalEncoding(d_model, max_len=(hzn_steps // patch) + 16)

        self.out_proj = nn.Linear(d_model, d_feats)

        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x):
        B, T, D = x.shape
        x_p = patchify(x, self.patch)
        z   = self.in_proj(x_p)
        z   = self.enc_pos(z)
        mem = self.encoder(z)

        dec_Tp = self.hzn_steps // self.patch
        tgt = torch.zeros(B, dec_Tp, self.d_model, device=x.device)
        tgt = self.dec_pos(tgt)
        out = self.decoder(tgt, mem)
        y_p = self.out_proj(out)
        y   = unpatchify(y_p, self.patch, self.hzn_steps)
        return y

In [ ]:
mse_loss = nn.MSELoss()

def rmse_only(pred, true):
    mse = mse_loss(pred, true)
    return torch.sqrt(mse + 1e-12)

def seq_rmse_np(y_true_np, y_pred_np):
    return float(np.sqrt(mean_squared_error(y_true_np.ravel(), y_pred_np.ravel())))

In [ ]:
# bim attack

def bim_attack(attacker_model,
               X,
               Y,
               eps,
               alpha,
               iters,
               random_start=False,
               return_grad_norm=False):

    attacker_model.eval()

    if random_start:
        # random uniform in [-eps, eps]
        noise = torch.empty_like(X).uniform_(-eps, eps)
        X_adv = (X + noise).clamp(0.0, 1.0)
    else:
        X_adv = X.clone()

    X_adv = X_adv.detach().requires_grad_(True)
    last_grad = None

    for _ in range(iters):
        preds = attacker_model(X_adv)
        loss  = mse_loss(preds, Y)

        if X_adv.grad is not None:
            X_adv.grad.zero_()

        loss.backward()
        grad = X_adv.grad
        last_grad = grad

        X_adv = (X_adv + alpha * grad.sign()).detach()
        delta = torch.clamp(X_adv - X, -eps, eps)
        X_adv = (X + delta).clamp(0.0, 1.0).detach().requires_grad_(True)

    X_adv = X_adv.detach()
    if return_grad_norm and last_grad is not None:
        gn = last_grad.view(last_grad.size(0), -1).norm(p=2, dim=1)
        return X_adv, gn.detach()
    return X_adv

In [ ]:
@torch.no_grad()
def eval_one_epoch(model, loader):
    model.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        p = model(xb)
        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())
    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

In [ ]:
def eval_under_attack(model_to_eval,
                      attacker_model,
                      loader,
                      atk_fn,
                      atk_kwargs):

    model_to_eval.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        xa = atk_fn(attacker_model, xb, yb, **atk_kwargs)

        with torch.no_grad():
            p = model_to_eval(xa)
        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())

    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

In [ ]:
def eval_bim_with_artifacts(model,
                            loader,
                            eps,
                            model_name,
                            run_i,
                            attack_name="bim",
                            iters=BIM_ITERS,
                            random_start=BIM_RANDOM_START_EVAL,
                            max_save_batches=1,
                            max_examples_per_batch=4):

    model.eval()

    H = HZN_STEPS
    D = 1

    alpha = eps / float(iters)

    sum_sq_clean = 0.0
    sum_sq_adv   = 0.0
    total_count  = 0

    sum_sq_clean_h = np.zeros(H, dtype=np.float64)
    sum_sq_adv_h   = np.zeros(H, dtype=np.float64)
    horizon_count  = 0

    grad_norm_rows = []

    sample_x = []
    sample_y = []
    sample_y_clean = []
    sample_y_adv   = []
    sample_x_adv   = []
    sample_delta   = []

    for batch_idx, (xb, yb) in enumerate(loader):
        xb = xb.to(device)
        yb = yb.to(device)

        xa, gn = bim_attack(
            model,
            xb,
            yb,
            eps=eps,
            alpha=alpha,
            iters=iters,
            random_start=random_start,
            return_grad_norm=True
        )

        with torch.no_grad():
            y_clean = model(xb)
            y_adv   = model(xa)

        # errors
        err_clean = (y_clean - yb) ** 2
        err_adv   = (y_adv   - yb) ** 2

        bs = xb.size(0)
        sum_sq_clean += err_clean.sum().item()
        sum_sq_adv   += err_adv.sum().item()
        total_count  += bs * H * D

        err_clean_np = err_clean.detach().cpu().numpy()
        err_adv_np   = err_adv.detach().cpu().numpy()

        sum_sq_clean_h += err_clean_np.sum(axis=(0, 2))
        sum_sq_adv_h   += err_adv_np.sum(axis=(0, 2))
        horizon_count  += bs * D

        # gradient norms
        gn_np = gn.detach().cpu().numpy()
        grad_norm_rows.append(pd.DataFrame({
            "run": run_i,
            "attack": attack_name,
            "epsilon": eps,
            "model": model_name,
            "batch_idx": batch_idx,
            "grad_norm": gn_np,
        }))

        # sample artifacts
        if batch_idx < max_save_batches:
            n_save = min(bs, max_examples_per_batch)
            xb_np      = xb[:n_save].detach().cpu().numpy()
            yb_np      = yb[:n_save].detach().cpu().numpy()
            yc_np      = y_clean[:n_save].detach().cpu().numpy()
            ya_np      = y_adv[:n_save].detach().cpu().numpy()
            xa_np      = xa[:n_save].detach().cpu().numpy()
            delta_np   = np.abs(ya_np - yc_np)

            sample_x.append(xb_np)
            sample_y.append(yb_np)
            sample_y_clean.append(yc_np)
            sample_y_adv.append(ya_np)
            sample_x_adv.append(xa_np)
            sample_delta.append(delta_np)

    # Aggregate grad norms
    if len(grad_norm_rows) > 0:
        df_gn = pd.concat(grad_norm_rows, ignore_index=True)
        append_df_to_csv(df_gn, GRAD_NORMS_CSV)

    # Global RMSE
    rmse_clean = math.sqrt(sum_sq_clean / max(1, total_count))
    rmse_adv   = math.sqrt(sum_sq_adv   / max(1, total_count))

    # Per-horizon RMSE
    rmse_clean_h = np.sqrt(sum_sq_clean_h / max(1, horizon_count))
    rmse_adv_h   = np.sqrt(sum_sq_adv_h   / max(1, horizon_count))

    df_ph = pd.DataFrame({
        "run": run_i,
        "attack": attack_name,
        "epsilon": eps,
        "model": model_name,
        "horizon_idx": np.arange(H),
        "rmse_clean": rmse_clean_h,
        "rmse_adv": rmse_adv_h,
    })
    append_df_to_csv(df_ph, PER_HORIZON_CSV)

    # save sample arrays + plots
    if len(sample_x) > 0:
        sx  = np.concatenate(sample_x, axis=0)
        sy  = np.concatenate(sample_y, axis=0)
        syc = np.concatenate(sample_y_clean, axis=0)
        sya = np.concatenate(sample_y_adv, axis=0)
        sxa = np.concatenate(sample_x_adv, axis=0)
        sdl = np.concatenate(sample_delta, axis=0)

        prefix = f"run{run_i:02d}_{model_name}_{attack_name}_eps{eps:.2f}"
        npz_path = os.path.join(ARTIFACTS_DIR, f"{prefix}_samples.npz")
        np.savez_compressed(
            npz_path,
            X=sx, Y=sy,
            Y_clean=syc,
            Y_adv=sya,
            X_adv=sxa,
            delta=sdl
        )

        t_hzn = np.arange(H)

        # GT vs clean vs adv
        ex_y   = sy[0, :, 0]
        ex_yc  = syc[0, :, 0]
        ex_ya  = sya[0, :, 0]

        plt.figure(figsize=(8, 4))
        plt.plot(t_hzn, ex_y,  label="GT")
        plt.plot(t_hzn, ex_yc, label="Clean pred")
        plt.plot(t_hzn, ex_ya, label="Adv pred")
        plt.xlabel("Horizon step")
        plt.ylabel("Scaled load")
        plt.title(f"GT vs Clean vs Adv | {prefix}")
        plt.legend()
        gt_plot = os.path.join(PLOTS_DIR, f"{prefix}_gt_clean_adv.png")
        plt.tight_layout()
        plt.savefig(gt_plot)
        plt.close()

        # Horizon-wise RMSE
        plt.figure(figsize=(8, 4))
        plt.plot(t_hzn, rmse_clean_h, label="Clean RMSE")
        plt.plot(t_hzn, rmse_adv_h,   label="Adv RMSE")
        plt.xlabel("Horizon step")
        plt.ylabel("RMSE")
        plt.title(f"Horizon-wise RMSE | {prefix}")
        plt.legend()
        hrmse_plot = os.path.join(PLOTS_DIR, f"{prefix}_horizon_rmse.png")
        plt.tight_layout()
        plt.savefig(hrmse_plot)
        plt.close()

        # Delta histogram
        plt.figure(figsize=(6, 4))
        plt.hist(sdl.flatten(), bins=50)
        plt.xlabel("|y_adv - y_clean|")
        plt.ylabel("Count")
        plt.title(f"Attack delta distribution | {prefix}")
        delta_plot = os.path.join(PLOTS_DIR, f"{prefix}_delta_hist.png")
        plt.tight_layout()
        plt.savefig(delta_plot)
        plt.close()

        # Input perturbation heatmap
        ex_x  = sx[0, :, 0]
        ex_xa = sxa[0, :, 0]
        pert  = ex_xa - ex_x
        plt.figure(figsize=(10, 2))
        plt.imshow(pert[None, :], aspect="auto", interpolation="nearest")
        plt.colorbar(label="delta X")
        plt.xlabel("History step")
        plt.yticks([])
        plt.title(f"Input perturbation heatmap | {prefix}")
        heatmap_plot = os.path.join(PLOTS_DIR, f"{prefix}_perturb_heatmap.png")
        plt.tight_layout()
        plt.savefig(heatmap_plot)
        plt.close()

        df_idx = pd.DataFrame([{
            "run": run_i,
            "model": model_name,
            "attack": attack_name,
            "epsilon": eps,
            "samples_npz": npz_path,
            "gt_clean_adv_plot": gt_plot,
            "horizon_rmse_plot": hrmse_plot,
            "delta_hist_plot": delta_plot,
            "perturb_heatmap_plot": heatmap_plot,
        }])
        append_df_to_csv(df_idx, ARTIFACT_INDEX_CSV)

    return rmse_clean, rmse_adv

In [ ]:
# load base

if not os.path.exists(BASE_CKPT):
    raise FileNotFoundError(f"Base checkpoint not found at {BASE_CKPT}")

print("Loading base checkpoint:", BASE_CKPT)

base_cpu = ElecSeq2SeqTransformer(
    d_feats=D_FEATS,
    d_model=D_MODEL,
    nhead=N_HEAD,
    num_layers=N_LAYERS,
    dim_ff=DIM_FF,
    dropout=DROPOUT,
    hist_steps=HIST_STEPS,
    hzn_steps=HZN_STEPS,
    patch=PATCH
).to("cpu")
base_cpu.load_state_dict(torch.load(BASE_CKPT, map_location="cpu"))

best_base = ElecSeq2SeqTransformer(
    d_feats=D_FEATS,
    d_model=D_MODEL,
    nhead=N_HEAD,
    num_layers=N_LAYERS,
    dim_ff=DIM_FF,
    dropout=DROPOUT,
    hist_steps=HIST_STEPS,
    hzn_steps=HZN_STEPS,
    patch=PATCH
).to(device)
best_base.load_state_dict(torch.load(BASE_CKPT, map_location=device))

# log base model size
n_params = model_n_params(base_cpu)
size_mb  = model_file_size_mb(BASE_CKPT)
df_ms = pd.DataFrame([{
    "model_name": "base_transformer_bim",
    "path": BASE_CKPT,
    "params": n_params,
    "size_mb": size_mb,
}])
append_df_to_csv(df_ms, MODEL_SIZES_CSV)

print("Base model loaded. Params:", n_params, " Size (MB):", size_mb)

Loading base checkpoint: /content/drive/MyDrive/298/Elec/SanityCheck/models/baseModel/elec_base_best.pth


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Base model loaded. Params: 3954433  Size (MB): 15.267773628234863


In [ ]:
# student generation

num_students = 4
student_paths = []

print("\ncreating students")
for i in range(1, num_students + 1):
    st = copy.deepcopy(base_cpu)
    with torch.no_grad():
        for _, p in st.named_parameters():
            p.add_(1e-3 * torch.randn_like(p))
    outp = os.path.join(STUDENTS_DIR, f"student_{i:02d}.pth")
    torch.save(st.state_dict(), outp)
    student_paths.append(outp)
    print("Saved student:", outp)

student_paths = sorted(student_paths)
print("Students:", student_paths)

# log student model sizes
for sp in student_paths:
    size_mb = model_file_size_mb(sp)
    df_ms = pd.DataFrame([{
        "model_name": os.path.basename(sp).replace(".pth", ""),
        "path": sp,
        "params": model_n_params(base_cpu),
        "size_mb": size_mb,
    }])
    append_df_to_csv(df_ms, MODEL_SIZES_CSV)



creating students
Saved student: /content/drive/MyDrive/298/Elec/SanityCheck/bim/students/student_01.pth
Saved student: /content/drive/MyDrive/298/Elec/SanityCheck/bim/students/student_02.pth
Saved student: /content/drive/MyDrive/298/Elec/SanityCheck/bim/students/student_03.pth
Saved student: /content/drive/MyDrive/298/Elec/SanityCheck/bim/students/student_04.pth
Students: ['/content/drive/MyDrive/298/Elec/SanityCheck/bim/students/student_01.pth', '/content/drive/MyDrive/298/Elec/SanityCheck/bim/students/student_02.pth', '/content/drive/MyDrive/298/Elec/SanityCheck/bim/students/student_03.pth', '/content/drive/MyDrive/298/Elec/SanityCheck/bim/students/student_04.pth']


In [ ]:
#bim adv training of students

adv_student_paths = []

print("\nbim adversarial training of students")

for sp in student_paths:
    idx = os.path.basename(sp).split("_")[1].split(".")[0]
    print(f"\n[Student {idx}] BIM adversarial training...")

    stu = ElecSeq2SeqTransformer(
        d_feats=D_FEATS,
        d_model=D_MODEL,
        nhead=N_HEAD,
        num_layers=N_LAYERS,
        dim_ff=DIM_FF,
        dropout=DROPOUT,
        hist_steps=HIST_STEPS,
        hzn_steps=HZN_STEPS,
        patch=PATCH
    ).to(device)
    stu.load_state_dict(torch.load(sp, map_location=device))

    opt = optim.AdamW(stu.parameters(), lr=LR_STUDENT)

    stu_history = []

    hb_flag, hb_thread = start_heartbeat(f"student_{idx}_bim_train")
    start_time = time.time()

    try:
        for ep in range(1, EPOCHS_STU + 1):
            stu.train()
            tot = cnt = 0
            for xb, yb in train_dl:
                xb = xb.to(device)
                yb = yb.to(device)

                # clean prediction
                pred_c = stu(xb)
                Lc = rmse_only(pred_c, yb)

                # multi epsilon bim adv training
                adv_losses = []
                for eps in BIM_TRAIN_EPS_LIST:
                    alpha = eps / float(BIM_ITERS)
                    xa = bim_attack(
                        stu, xb, yb,
                        eps=eps,
                        alpha=alpha,
                        iters=BIM_ITERS,
                        random_start=BIM_RANDOM_START_TRAIN
                    )
                    pred_a = stu(xa)
                    La = rmse_only(pred_a, yb)
                    adv_losses.append(La)

                if len(adv_losses) > 0:
                    La_mean = torch.stack(adv_losses).mean()
                else:
                    La_mean = 0.0 * Lc

                loss = 0.25 * Lc + 0.75 * La_mean

                opt.zero_grad()
                loss.backward()
                opt.step()

                bs = xb.size(0)
                tot += loss.item() * bs
                cnt += bs

            avg_loss = tot / max(1, cnt)
            elapsed  = (time.time() - start_time) / 60.0
            print(f"  [Student {idx}] Epoch {ep:02d} | avg_loss={avg_loss:.4f} | elapsed={elapsed:.2f} min")

            stu_history.append({
                "student_idx": int(idx),
                "epoch": ep,
                "avg_loss": avg_loss,
                "elapsed_min": elapsed,
            })

    except Exception as e:
        # create log on crash and keep a checkpoint
        stop_heartbeat(hb_flag, hb_thread)
        msg = f"[Student {idx}] BIM adv training crash: {repr(e)}\n{traceback.format_exc()}"
        print(msg)
        log_error(msg)
        ckpt_crash = os.path.join(STUDENTS_ADV_DIR, f"student_{idx}_bim_CRASHED.pth")
        torch.save(stu.to("cpu").state_dict(), ckpt_crash)
        raise

    finally:
        stop_heartbeat(hb_flag, hb_thread)

    # save per student training history csv
    stu_hist_path = os.path.join(RESULTS_DIR, f"student_{idx}_bim_train_history.csv")
    df_stu_hist = pd.DataFrame(stu_history)
    df_stu_hist.to_csv(stu_hist_path, index=False)
    print(f"  [Student {idx}] Saved BIM training history to:", stu_hist_path)

    outp = os.path.join(STUDENTS_ADV_DIR, f"student_{idx}_bim_adv.pth")
    torch.save(stu.to("cpu").state_dict(), outp)
    adv_student_paths.append(outp)
    print(f"  [Student {idx}] Saved BIM adv student:", outp)

    # log adv student model size
    size_mb = model_file_size_mb(outp)
    df_ms = pd.DataFrame([{
        "model_name": f"student_{idx}_bim_adv",
        "path": outp,
        "params": model_n_params(base_cpu),
        "size_mb": size_mb,
    }])
    append_df_to_csv(df_ms, MODEL_SIZES_CSV)

adv_student_paths = sorted(adv_student_paths)
print("\nBIM adv students:", adv_student_paths)


bim adversarial training of students

[Student 01] BIM adversarial training...
[heartbeat:student_01_bim_train] running... elapsed=10.00 min
[heartbeat:student_01_bim_train] running... elapsed=20.00 min
  [Student 01] Epoch 01 | avg_loss=0.1450 | elapsed=22.30 min
[heartbeat:student_01_bim_train] running... elapsed=30.00 min
[heartbeat:student_01_bim_train] running... elapsed=40.00 min
  [Student 01] Epoch 02 | avg_loss=0.1426 | elapsed=44.74 min
[heartbeat:student_01_bim_train] running... elapsed=50.00 min
[heartbeat:student_01_bim_train] running... elapsed=60.00 min
  [Student 01] Epoch 03 | avg_loss=0.1379 | elapsed=67.23 min
[heartbeat:student_01_bim_train] running... elapsed=70.00 min
[heartbeat:student_01_bim_train] running... elapsed=80.00 min
  [Student 01] Epoch 04 | avg_loss=0.1338 | elapsed=89.08 min
[heartbeat:student_01_bim_train] running... elapsed=90.00 min
[heartbeat:student_01_bim_train] running... elapsed=100.00 min
[heartbeat:student_01_bim_train] running... elapsed

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


[heartbeat:student_02_bim_train] running... elapsed=10.00 min
[heartbeat:student_02_bim_train] running... elapsed=20.00 min
  [Student 02] Epoch 01 | avg_loss=0.1450 | elapsed=21.56 min
[heartbeat:student_02_bim_train] running... elapsed=30.00 min
[heartbeat:student_02_bim_train] running... elapsed=40.00 min
  [Student 02] Epoch 02 | avg_loss=0.1421 | elapsed=43.19 min
[heartbeat:student_02_bim_train] running... elapsed=50.00 min
[heartbeat:student_02_bim_train] running... elapsed=60.00 min
  [Student 02] Epoch 03 | avg_loss=0.1371 | elapsed=65.07 min
[heartbeat:student_02_bim_train] running... elapsed=70.00 min
[heartbeat:student_02_bim_train] running... elapsed=80.00 min
  [Student 02] Epoch 04 | avg_loss=0.1320 | elapsed=86.85 min
[heartbeat:student_02_bim_train] running... elapsed=90.00 min
[heartbeat:student_02_bim_train] running... elapsed=100.00 min
  [Student 02] Epoch 05 | avg_loss=0.1298 | elapsed=108.56 min
  [Student 02] Saved BIM training history to: /content/drive/MyDrive

In [ ]:
BIM_SANITY_DIR           = "/content/drive/MyDrive/298/Elec/SanityCheck/bim"
BIM_STUDENTS_ADV_DIR     = os.path.join(BIM_SANITY_DIR, "students_adv")

BIM_FINAL_PROJECT_DIR    = "/content/drive/MyDrive/298/Elec/FinalPipeline/bim"
BIM_FINAL_RESULTS_DIR    = os.path.join(BIM_FINAL_PROJECT_DIR, "results")
BIM_FINAL_PLOTS_DIR      = os.path.join(BIM_FINAL_RESULTS_DIR, "plots")
BIM_FINAL_ARTIFACTS_DIR  = os.path.join(BIM_FINAL_RESULTS_DIR, "artifacts")

for d in [BIM_FINAL_PROJECT_DIR, BIM_FINAL_RESULTS_DIR, BIM_FINAL_PLOTS_DIR, BIM_FINAL_ARTIFACTS_DIR]:
    os.makedirs(d, exist_ok=True)

BIM_RUNS_CSV             = os.path.join(BIM_FINAL_RESULTS_DIR, "bim_morphence_runs.csv")
BIM_STATS_CSV            = os.path.join(BIM_FINAL_RESULTS_DIR, "bim_morphence_stats.csv")
BIM_SINGLE_RUN_CSV       = os.path.join(BIM_FINAL_RESULTS_DIR, "bim_morphence_single_run.csv")
BIM_XFER_MATRIX_CSV      = os.path.join(BIM_FINAL_RESULTS_DIR, "bim_transfer_matrix.csv")
BIM_WEIGHT_DIVERSITY_CSV = os.path.join(BIM_FINAL_RESULTS_DIR, "bim_weight_diversity.csv")
BIM_PRED_DIVERSITY_CSV   = os.path.join(BIM_FINAL_RESULTS_DIR, "bim_prediction_diversity.csv")

print("\n[BIM] SanityCheck BIM dir:", BIM_SANITY_DIR)
print("[BIM] BIM students_adv dir:", BIM_STUDENTS_ADV_DIR)
print("[BIM] FinalPipeline BIM results dir:", BIM_FINAL_RESULTS_DIR)


[BIM] SanityCheck BIM dir: /content/drive/MyDrive/298/Elec/SanityCheck/bim
[BIM] BIM students_adv dir: /content/drive/MyDrive/298/Elec/SanityCheck/bim/students_adv
[BIM] FinalPipeline BIM results dir: /content/drive/MyDrive/298/Elec/FinalPipeline/bim/results


In [ ]:
BIM_EVAL_EPS_LIST     = [0.1, 0.2, 0.3, 0.5]
BIM_ITERS             = 10
BIM_RANDOM_START_EVAL = True

In [ ]:
def bim_attack(attacker_model,
               X,
               Y,
               eps,
               alpha,
               iters,
               random_start=False):

    attacker_model.eval()

    if random_start:
        # Random start in [-eps, eps]
        noise = torch.empty_like(X).uniform_(-eps, eps)
        X_adv = (X + noise).clamp(0.0, 1.0)
    else:
        X_adv = X.clone()

    X_adv = X_adv.detach().requires_grad_(True)

    for _ in range(iters):
        preds = attacker_model(X_adv)
        loss  = mse_loss(preds, Y)

        if X_adv.grad is not None:
            X_adv.grad.zero_()

        loss.backward()
        grad = X_adv.grad

        X_adv = (X_adv + alpha * grad.sign()).detach()
        delta = torch.clamp(X_adv - X, -eps, eps)
        X_adv = (X + delta).clamp(0.0, 1.0).detach().requires_grad_(True)

    return X_adv.detach()

In [ ]:
bim_student_paths = sorted(
    [
        os.path.join(BIM_STUDENTS_ADV_DIR, f)
        for f in os.listdir(BIM_STUDENTS_ADV_DIR)
        if f.endswith("_bim_adv.pth")
    ]
)

assert len(bim_student_paths) > 0, "[BIM] No *_bim_adv.pth found in BIM students_adv dir!"
print("\n[BIM] Adv students (from SanityCheck/bim):")
for p in bim_student_paths:
    print("  ", p)

bim_adv_students = []
for sp in bim_student_paths:
    stu = ElecSeq2SeqTransformer(
        d_feats=D_FEATS,
        d_model=D_MODEL,
        nhead=N_HEAD,
        num_layers=N_LAYERS,
        dim_ff=DIM_FF,
        dropout=DROPOUT,
        hist_steps=HIST_STEPS,
        hzn_steps=HZN_STEPS,
        patch=PATCH
    ).to(device)
    stu.load_state_dict(torch.load(sp, map_location=device))
    stu.eval()
    bim_adv_students.append(stu)

bim_student_names = [os.path.basename(p).replace(".pth", "") for p in bim_student_paths]
print(f"[BIM] Loaded {len(bim_adv_students)} BIM adversarially trained students.")


[BIM] Adv students (from SanityCheck/bim):
   /content/drive/MyDrive/298/Elec/SanityCheck/bim/students_adv/student_01_bim_adv.pth
   /content/drive/MyDrive/298/Elec/SanityCheck/bim/students_adv/student_02_bim_adv.pth
   /content/drive/MyDrive/298/Elec/SanityCheck/bim/students_adv/student_03_bim_adv.pth
   /content/drive/MyDrive/298/Elec/SanityCheck/bim/students_adv/student_04_bim_adv.pth
[BIM] Loaded 4 BIM adversarially trained students.


In [ ]:
def eval_pair_bim(defender, attacker, loader, eps,
                  iters=BIM_ITERS,
                  random_start=BIM_RANDOM_START_EVAL):
    defender.eval()
    attacker.eval()
    alpha = eps / float(iters)

    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        xa = bim_attack(
            attacker,
            xb,
            yb,
            eps=eps,
            alpha=alpha,
            iters=iters,
            random_start=random_start,
        )
        with torch.no_grad():
            p = defender(xa)

        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())

    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

In [ ]:
def eval_random_switch_bim(models, loader, eps,
                           iters=BIM_ITERS,
                           random_start=BIM_RANDOM_START_EVAL):
    for m in models:
        m.eval()

    ys, ps = [], []
    n_stu = len(models)
    alpha = eps / float(iters)

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        atk_idx  = random.randrange(n_stu)
        def_idx  = random.randrange(n_stu)
        attacker = models[atk_idx]
        defender = models[def_idx]

        xa = bim_attack(
            attacker,
            xb,
            yb,
            eps=eps,
            alpha=alpha,
            iters=iters,
            random_start=random_start,
        )
        with torch.no_grad():
            p = defender(xa)

        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())

    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

In [ ]:
def compute_weight_diversity_bim(students, names):
    vecs = [flatten_params(m) for m in students]
    k = len(vecs)
    rows = []
    for i, j in itertools.combinations(range(k), 2):
        d = torch.norm(vecs[i] - vecs[j], p=2).item()
        rows.append({
            "student_i": names[i],
            "student_j": names[j],
            "l2_distance": d,
        })
    df_w = pd.DataFrame(rows)
    df_w.to_csv(BIM_WEIGHT_DIVERSITY_CSV, index=False)
    print("[BIM] Saved weight diversity to:", BIM_WEIGHT_DIVERSITY_CSV)
    return df_w

In [ ]:
@torch.no_grad()
def compute_prediction_diversity_bim(students, loader):
    for m in students:
        m.eval()

    k = len(students)
    H = HZN_STEPS

    sum_pred  = np.zeros(H, dtype=np.float64)
    sum_pred2 = np.zeros(H, dtype=np.float64)
    count     = 0

    for xb, yb in loader:
        xb = xb.to(device)
        preds_stu = []
        for m in students:
            preds_stu.append(m(xb).cpu().numpy())  # (B,H,1)
        preds_stu = np.stack(preds_stu, axis=0)  # (k,B,H,1)

        preds_mean_batch = preds_stu.mean(axis=1).squeeze(-1)  # (k,H)

        sum_pred  += preds_mean_batch.mean(axis=0)
        sum_pred2 += (preds_mean_batch ** 2).mean(axis=0)
        count     += 1

    mean_pred  = sum_pred / max(1, count)
    mean_pred2 = sum_pred2 / max(1, count)
    var_pred   = np.maximum(mean_pred2 - mean_pred**2, 0.0)

    df_p = pd.DataFrame({
        "horizon_idx": np.arange(H),
        "mean_pred": mean_pred,
        "var_pred": var_pred,
    })
    df_p.to_csv(BIM_PRED_DIVERSITY_CSV, index=False)
    print("[BIM] Saved prediction diversity to:", BIM_PRED_DIVERSITY_CSV)
    return df_p

print("\n[BIM] Computing diversity metrics (once)...")
_ = compute_weight_diversity_bim(bim_adv_students, bim_student_names)
_ = compute_prediction_diversity_bim(bim_adv_students, test_dl)


[BIM] Computing diversity metrics (once)...
[BIM] Saved weight diversity to: /content/drive/MyDrive/298/Elec/FinalPipeline/bim/results/bim_weight_diversity.csv
[BIM] Saved prediction diversity to: /content/drive/MyDrive/298/Elec/FinalPipeline/bim/results/bim_prediction_diversity.csv


In [ ]:
bim_xfer_rows = []
for eps in BIM_EVAL_EPS_LIST:
    print(f"\n[BIM] Computing transfer matrix for eps={eps}...")
    for i, atk_name in enumerate(bim_student_names):
        for j, def_name in enumerate(bim_student_names):
            atk_model = bim_adv_students[i]
            def_model = bim_adv_students[j]
            rmse_ij = eval_pair_bim(def_model, atk_model, test_dl, eps=eps)
            bim_xfer_rows.append({
                "epsilon": eps,
                "attacker": atk_name,
                "defender": def_name,
                "rmse_adv": rmse_ij,
            })
            print(f"  atk={atk_name} -> def={def_name} | eps={eps:.2f} | RMSE={rmse_ij:.5f}")

df_bim_xfer = pd.DataFrame(bim_xfer_rows)
df_bim_xfer.to_csv(BIM_XFER_MATRIX_CSV, index=False)
print("\n[BIM] Saved BIM transfer matrix to:", BIM_XFER_MATRIX_CSV)



[BIM] Computing transfer matrix for eps=0.1...
  atk=student_01_bim_adv -> def=student_01_bim_adv | eps=0.10 | RMSE=0.18414
  atk=student_01_bim_adv -> def=student_02_bim_adv | eps=0.10 | RMSE=0.18812
  atk=student_01_bim_adv -> def=student_03_bim_adv | eps=0.10 | RMSE=0.19763
  atk=student_01_bim_adv -> def=student_04_bim_adv | eps=0.10 | RMSE=0.20170
  atk=student_02_bim_adv -> def=student_01_bim_adv | eps=0.10 | RMSE=0.18020
  atk=student_02_bim_adv -> def=student_02_bim_adv | eps=0.10 | RMSE=0.19182
  atk=student_02_bim_adv -> def=student_03_bim_adv | eps=0.10 | RMSE=0.19874
  atk=student_02_bim_adv -> def=student_04_bim_adv | eps=0.10 | RMSE=0.20214
  atk=student_03_bim_adv -> def=student_01_bim_adv | eps=0.10 | RMSE=0.17986
  atk=student_03_bim_adv -> def=student_02_bim_adv | eps=0.10 | RMSE=0.18934
  atk=student_03_bim_adv -> def=student_03_bim_adv | eps=0.10 | RMSE=0.20204
  atk=student_03_bim_adv -> def=student_04_bim_adv | eps=0.10 | RMSE=0.20123
  atk=student_04_bim_adv -> 

In [ ]:
print("\n[BIM] Single run BIM eval: base vs ensemble")
print("[BIM] Epsilons:", BIM_EVAL_EPS_LIST)

bim_single_seed = 2233
random.seed(bim_single_seed)
np.random.seed(bim_single_seed)
torch.manual_seed(bim_single_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(bim_single_seed)

bim_single_rows = []   # ["attack", "epsilon", "model", "RMSE"]

# clean performance (reuse best_base + test_dl)
bim_base_clean_single   = eval_clean_rmse(best_base,  test_dl)
bim_morph_clean_single  = eval_ensemble_rmse(bim_adv_students, test_dl)

bim_single_rows.append(["clean", None, "base",           bim_base_clean_single])
bim_single_rows.append(["clean", None, "morph_ensemble", bim_morph_clean_single])

print(
    f"[BIM Single-run] clean | base={bim_base_clean_single:.5f} | "
    f"morph_ensemble={bim_morph_clean_single:.5f}",
    flush=True
)

for eps in BIM_EVAL_EPS_LIST:
    bim_base_adv_single = eval_pair_bim(
        defender=best_base,
        attacker=best_base,
        loader=test_dl,
        eps=eps
    )

    bim_morph_rand_single = eval_random_switch_bim(
        models=bim_adv_students,
        loader=test_dl,
        eps=eps
    )

    bim_single_rows.append(["bim", eps, "base",        bim_base_adv_single])
    bim_single_rows.append(["bim", eps, "morph_rand",  bim_morph_rand_single])

    print(
        f"[BIM Single run] eps={eps:.2f} | base_adv={bim_base_adv_single:.5f} | "
        f"morph_rand={bim_morph_rand_single:.5f}",
        flush=True
    )

df_bim_single = pd.DataFrame(
    bim_single_rows,
    columns=["attack", "epsilon", "model", "RMSE"]
)
df_bim_single.to_csv(BIM_SINGLE_RUN_CSV, index=False)
print("\n[BIM] Saved single run BIM eval CSV to:", BIM_SINGLE_RUN_CSV)


[BIM] Single run BIM eval: base vs ensemble
[BIM] Epsilons: [0.1, 0.2, 0.3, 0.5]
[BIM Single-run] clean | base=0.11506 | morph_ensemble=0.15825
[BIM Single run] eps=0.10 | base_adv=0.24913 | morph_rand=0.19339
[BIM Single run] eps=0.20 | base_adv=0.33837 | morph_rand=0.19537
[BIM Single run] eps=0.30 | base_adv=0.36804 | morph_rand=0.20060
[BIM Single run] eps=0.50 | base_adv=0.39547 | morph_rand=0.24415

[BIM] Saved single run BIM eval CSV to: /content/drive/MyDrive/298/Elec/FinalPipeline/bim/results/bim_morphence_single_run.csv


In [ ]:
print("\n[BIM] 30 run BIM eval: base vs ensemble")
print("[BIM] Epsilons:", BIM_EVAL_EPS_LIST)
print("[BIM] Runs:", N_RUNS)

bim_all_rows = []   # ["attack", "epsilon", "model", "RMSE", "run"]

bim_start_all = time.time()
bim_hb_flag, bim_hb_thread = start_heartbeat("eval_bim_morphence")

try:
    for run_i in range(1, N_RUNS + 1):
        run_start = time.time()
        seed_i = 7000 + run_i
        random.seed(seed_i)
        np.random.seed(seed_i)
        torch.manual_seed(seed_i)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed_i)

        # clean
        base_clean  = eval_clean_rmse(best_base, test_dl)
        morph_clean = eval_ensemble_rmse(bim_adv_students, test_dl)

        bim_all_rows.append(["clean", None, "base",           base_clean,  run_i])
        bim_all_rows.append(["clean", None, "morph_ensemble", morph_clean, run_i])

        print(f"[BIM Run {run_i:02d}] clean | base={base_clean:.5f} | morph_ens={morph_clean:.5f}",
              flush=True)

        # BIM robustness (white-box base, random-switch Morphence)
        for eps in BIM_EVAL_EPS_LIST:
            base_adv   = eval_pair_bim(best_base, best_base, test_dl, eps=eps)
            morph_rand = eval_random_switch_bim(bim_adv_students, test_dl, eps=eps)

            bim_all_rows.append(["bim", eps, "base",       base_adv,   run_i])
            bim_all_rows.append(["bim", eps, "morph_rand", morph_rand, run_i])

            print(
                f"[BIM Run {run_i:02d}] eps={eps:.2f} | base_adv={base_adv:.5f} | "
                f"morph_rand={morph_rand:.5f}",
                flush=True
            )

        # autosave after each run
        df_bim_runs = pd.DataFrame(
            bim_all_rows,
            columns=["attack", "epsilon", "model", "RMSE", "run"]
        )
        df_bim_runs.to_csv(BIM_RUNS_CSV, index=False)

        bim_stats = (
            df_bim_runs
            .groupby(["attack", "epsilon", "model"])["RMSE"]
            .agg(mean="mean", std="std", n="count")
            .reset_index()
            .sort_values(["attack", "epsilon", "model"])
        )
        bim_stats.to_csv(BIM_STATS_CSV, index=False)

        run_mins   = (time.time() - run_start) / 60.0
        total_mins = (time.time() - bim_start_all) / 60.0
        print(
            f"[BIM] Completed run {run_i}/{N_RUNS} | "
            f"run_time={run_mins:.2f} min | total_elapsed={total_mins:.2f} min",
            flush=True
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        gc.collect()

finally:
    stop_heartbeat(bim_hb_flag, bim_hb_thread)

print("\n[BIM] Saved BIM ensemble eval runs CSV to:", BIM_RUNS_CSV)
print("[BIM] Saved BIM ensemble eval stats CSV to:", BIM_STATS_CSV)
print("[BIM] Weight diversity CSV:", BIM_WEIGHT_DIVERSITY_CSV)
print("[BIM] Prediction diversity CSV:", BIM_PRED_DIVERSITY_CSV)
print("[BIM] Transfer matrix CSV:", BIM_XFER_MATRIX_CSV)
print("[BIM] FinalPipeline BIM plots dir:", BIM_FINAL_PLOTS_DIR)
print("[BIM] FinalPipeline BIM artifacts dir:", BIM_FINAL_ARTIFACTS_DIR)


[BIM] 30 run BIM eval: base vs ensemble
[BIM] Epsilons: [0.1, 0.2, 0.3, 0.5]
[BIM] Runs: 30
[BIM Run 01] clean | base=0.11506 | morph_ens=0.15825
[BIM Run 01] eps=0.10 | base_adv=0.24959 | morph_rand=0.19297
[BIM Run 01] eps=0.20 | base_adv=0.33807 | morph_rand=0.19745
[BIM Run 01] eps=0.30 | base_adv=0.36743 | morph_rand=0.20078
[heartbeat:eval_bim_morphence] running... elapsed=10.00 min
[BIM Run 01] eps=0.50 | base_adv=0.39610 | morph_rand=0.24401
[BIM] Completed run 1/30 | run_time=11.57 min | total_elapsed=11.57 min
[BIM Run 02] clean | base=0.11506 | morph_ens=0.15825
[BIM Run 02] eps=0.10 | base_adv=0.24992 | morph_rand=0.19253
[BIM Run 02] eps=0.20 | base_adv=0.33879 | morph_rand=0.19624
[heartbeat:eval_bim_morphence] running... elapsed=20.00 min
[BIM Run 02] eps=0.30 | base_adv=0.36790 | morph_rand=0.20034
[BIM Run 02] eps=0.50 | base_adv=0.39639 | morph_rand=0.24458
[BIM] Completed run 2/30 | run_time=11.49 min | total_elapsed=23.06 min
[BIM Run 03] clean | base=0.11506 | mor

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np

# recreate Elec FGSM / BIM result paths

ELEC_BASE_DIR = "/content/drive/MyDrive/298/Elec/FinalPipeline"

FGSM_RESULTS_DIR = os.path.join(ELEC_BASE_DIR, "fgsm", "results")
BIM_RESULTS_DIR  = os.path.join(ELEC_BASE_DIR, "bim",  "results")

FGSM_RUNS_CSV = os.path.join(FGSM_RESULTS_DIR, "fgsm_morphence_runs.csv")
BIM_RUNS_CSV  = os.path.join(BIM_RESULTS_DIR,  "bim_morphence_runs.csv")

print("FGSM runs CSV:", FGSM_RUNS_CSV)
print("BIM  runs CSV:", BIM_RUNS_CSV)

os.makedirs(FGSM_RESULTS_DIR, exist_ok=True)
os.makedirs(BIM_RESULTS_DIR,  exist_ok=True)

# config for both attacks
ATTACK_CONFIGS = [
    dict(
        name="FGSM",
        prefix="fgsm",
        attack_key="rfgsm",
        runs_csv=FGSM_RUNS_CSV,
        results_dir=FGSM_RESULTS_DIR,
    ),
    dict(
        name="BIM",
        prefix="bim",
        attack_key="bim",
        runs_csv=BIM_RUNS_CSV,
        results_dir=BIM_RESULTS_DIR,
    ),
]

# compute stats for one attack (FGSM or BIM)
def compute_stats_for_attack(cfg):
    """
    Expects a runs CSV with columns:
      attack, epsilon, model, RMSE, run
    and 'attack' values: 'clean' + cfg['attack_key'] (e.g., 'rfgsm', 'bim').
    """
    name        = cfg["name"]
    prefix      = cfg["prefix"]
    attack_key  = cfg["attack_key"]
    runs_csv    = cfg["runs_csv"]
    results_dir = cfg["results_dir"]

    print("\n==============================================")
    print(f"Processing {name} runs")
    print(f"Runs CSV: {runs_csv}")
    print("==============================================")

    if not os.path.exists(runs_csv):
        print(f"[WARN] Runs CSV not found for {name}: {runs_csv}")
        return

    df_runs = pd.read_csv(runs_csv)
    print("Loaded runs:", df_runs.shape)
    print(df_runs.head())

    # clean stats (base vs ensemble)
    df_clean = df_runs[df_runs["attack"] == "clean"].copy()

    df_clean["model"] = pd.Categorical(
        df_clean["model"],
        categories=["base", "morph_ensemble"],
        ordered=True,
    )

    clean_stats = (
        df_clean
        .groupby("model", observed=False)["RMSE"]
        .agg(
            mean   ="mean",
            std    ="std",
            min    ="min",
            q1     =lambda s: s.quantile(0.25),
            median ="median",
            q3     =lambda s: s.quantile(0.75),
            max    ="max",
            n      ="count",
        )
        .reset_index()
        .sort_values("model")
    )

    print(f"\n[{name} | Clean] stats:")
    print(clean_stats.to_string(index=False))

    clean_csv = os.path.join(results_dir, f"{prefix}_clean_stats.csv")
    clean_stats.to_csv(clean_csv, index=False)
    print("Saved clean stats to:", clean_csv)

    # attack stats (base vs morph_rand)
    df_attack = df_runs[df_runs["attack"] == attack_key].copy()

    def model_attack_stats(df, model_name):
        st = (
            df[df["model"] == model_name]
            .groupby("epsilon", observed=False)["RMSE"]
            .agg(
                mean   ="mean",
                std    ="std",
                min    ="min",
                q1     =lambda s: s.quantile(0.25),
                median ="median",
                q3     =lambda s: s.quantile(0.75),
                max    ="max",
                n      ="count",
            )
            .reset_index()
            .sort_values("epsilon")
        )

        print(f"\n[{name} | {attack_key}] {model_name} stats:")
        print(st.to_string(index=False))

        out_csv = os.path.join(
            results_dir, f"{prefix}_{attack_key}_{model_name}_stats.csv"
        )
        st.to_csv(out_csv, index=False)
        print(f"Saved: {out_csv}")

    # base white-box stats
    model_attack_stats(df_attack, "base")

    # Morphence ensemble under attack (random switching)
    model_attack_stats(df_attack, "morph_rand")


# run for both FGSM and BIM
for cfg in ATTACK_CONFIGS:
    compute_stats_for_attack(cfg)


Mounted at /content/drive
FGSM runs CSV: /content/drive/MyDrive/298/Elec/FinalPipeline/fgsm/results/fgsm_morphence_runs.csv
BIM  runs CSV: /content/drive/MyDrive/298/Elec/FinalPipeline/bim/results/bim_morphence_runs.csv

Processing FGSM runs
Runs CSV: /content/drive/MyDrive/298/Elec/FinalPipeline/fgsm/results/fgsm_morphence_runs.csv
Loaded runs: (300, 5)
  attack  epsilon           model      RMSE  run
0  clean      NaN            base  0.115062    1
1  clean      NaN  morph_ensemble  0.117630    1
2  rfgsm      0.1            base  0.169737    1
3  rfgsm      0.1      morph_rand  0.142028    1
4  rfgsm      0.2            base  0.229181    1

[FGSM | Clean] stats:
         model     mean  std      min       q1   median       q3      max  n
          base 0.115062  0.0 0.115062 0.115062 0.115062 0.115062 0.115062 30
morph_ensemble 0.117630  0.0 0.117630 0.117630 0.117630 0.117630 0.117630 30
Saved clean stats to: /content/drive/MyDrive/298/Elec/FinalPipeline/fgsm/results/fgsm_clean_sta

<h1>PGD</h1>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, math, copy, random, time, threading, gc, traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


In [ ]:
DATA_PATH = "/content/drive/MyDrive/298/Elec/Data/LD2011_2014.txt"

BASE_MODEL_DIR = "/content/drive/MyDrive/298/Elec/SanityCheck/models/baseModel"
BASE_CKPT      = os.path.join(BASE_MODEL_DIR, "elec_base_best.pth")

PROJECT_DIR    = "/content/drive/MyDrive/298/Elec/SanityCheck/pgd"

STUDENTS_DIR     = os.path.join(PROJECT_DIR, "students")
STUDENTS_ADV_DIR = os.path.join(PROJECT_DIR, "students_adv")
RESULTS_DIR      = os.path.join(PROJECT_DIR, "results")
PLOTS_DIR        = os.path.join(RESULTS_DIR, "plots")
ARTIFACTS_DIR    = os.path.join(RESULTS_DIR, "artifacts")
CRASH_DIR        = os.path.join(PROJECT_DIR, "crash_dump")

for d in [PROJECT_DIR, STUDENTS_DIR, STUDENTS_ADV_DIR, RESULTS_DIR, PLOTS_DIR, ARTIFACTS_DIR, CRASH_DIR]:
    os.makedirs(d, exist_ok=True)

RUNS_CSV          = os.path.join(RESULTS_DIR, "pgd_runs.csv")
STATS_CSV         = os.path.join(RESULTS_DIR, "pgd_stats.csv")
PER_HORIZON_CSV   = os.path.join(RESULTS_DIR, "pgd_per_horizon_rmse.csv")
GRAD_NORMS_CSV    = os.path.join(RESULTS_DIR, "pgd_grad_norms.csv")
MODEL_SIZES_CSV   = os.path.join(RESULTS_DIR, "pgd_model_sizes.csv")
ARTIFACT_INDEX_CSV= os.path.join(RESULTS_DIR, "pgd_artifact_index.csv")
ERROR_LOG         = os.path.join(CRASH_DIR, "error_log.txt")

In [ ]:
FREQ = "15min"
STEPS_PER_HOUR = 4
HOURS_PER_DAY  = 24

HIST_HOURS  = 96
HZN_HOURS   = 24
HIST_STEPS  = HIST_HOURS * STEPS_PER_HOUR   # 384
HZN_STEPS   = HZN_HOURS  * STEPS_PER_HOUR   # 96
SLIDE       = 1

TRAIN_FRAC  = 0.70
VAL_FRAC    = 0.10

BATCH       = 16
PATCH       = 4
D_FEATS     = 1
D_MODEL     = 256
N_HEAD      = 4
N_LAYERS    = 3
DIM_FF      = 512
DROPOUT     = 0.1

LR_STUDENT  = 1e-4
EPOCHS_STU  = 5

In [ ]:
PGD_TRAIN_EPS_LIST = [0.10, 0.20, 0.30, 0.50]
PGD_EVAL_EPS_LIST  = [0.10, 0.20, 0.30, 0.50]
PGD_ITERS          = 10

# random start choices:
PGD_RANDOM_START_TRAIN = True
PGD_RANDOM_START_EVAL  = True

N_RUNS = 30
HEARTBEAT_SECS = 600  # 10 minutes

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", device)

Device: cpu


In [ ]:
def start_heartbeat(name="task"):
    flag = {"on": True}
    start_time = time.time()

    def _hb():
        while flag["on"]:
            time.sleep(HEARTBEAT_SECS)
            if flag["on"]:
                elapsed = (time.time() - start_time) / 60.0
                print(f"[heartbeat:{name}] running... elapsed={elapsed:.2f} min", flush=True)

    t = threading.Thread(target=_hb, daemon=True)
    t.start()
    return flag, t

def stop_heartbeat(flag, thread):
    flag["on"] = False
    try:
        thread.join(timeout=1)
    except Exception:
        pass

def append_df_to_csv(df, path):
    header = not os.path.exists(path)
    df.to_csv(path, mode="a", header=header, index=False)

def model_n_params(model: nn.Module):
    return sum(p.numel() for p in model.parameters())

def model_file_size_mb(path: str):
    if not os.path.exists(path):
        return None
    return os.path.getsize(path) / (1024.0 * 1024.0)

def log_error(msg: str):
    with open(ERROR_LOG, "a") as f:
        f.write(msg + "\n")

In [ ]:
print("Reading raw txt:", DATA_PATH)
df = pd.read_csv(
    DATA_PATH,
    sep=';',
    quotechar='"',
    decimal=',',
    parse_dates=[0],
    index_col=0,
    na_values=['', 'NA', 'NaN', ''],
    low_memory=False
)
df.index.name = "timestamp"
df = df.sort_index()
print("Raw shape:", df.shape)

METER = "MT_001"
print("Using single meter:", METER)

START_DT = pd.Timestamp("2013-01-01 00:00:00")
END_DT   = pd.Timestamp("2013-12-31 23:45:00")

df = df[[METER]]
df = df.loc[(df.index >= START_DT) & (df.index <= END_DT)].copy()

# regularize index
full_idx = pd.date_range(START_DT, END_DT, freq=FREQ)
df = df.reindex(full_idx)

# fill small gaps
df = df.apply(pd.to_numeric, errors='coerce')
df = df.ffill(limit=4).bfill(limit=4)
print("Post filtering shape:", df.shape)

# scale on early TRAIN_FRAC portion of time
n_t       = len(df)
split_idx = int(TRAIN_FRAC * n_t)
df_train  = df.iloc[:split_idx]
df_all    = df

scaler = MinMaxScaler()
train_scaled = pd.DataFrame(
    scaler.fit_transform(df_train),
    index=df_train.index,
    columns=df_train.columns
)
full_scaled = pd.DataFrame(
    scaler.transform(df_all),
    index=df_all.index,
    columns=df_all.columns
)
print("Scaled shape:", full_scaled.shape)

Reading raw txt: /content/drive/MyDrive/298/Elec/Data/LD2011_2014.txt
Raw shape: (140256, 370)
Using single meter: MT_001
Post filtering shape: (35040, 1)
Scaled shape: (35040, 1)


In [ ]:
values = full_scaled.values.astype(np.float32)  # (T, 1)
T, D = values.shape
assert D == 1
print("T, D:", T, D)

X_list, Y_list, meta = [], [], []
for t in range(HIST_STEPS, T - HZN_STEPS, SLIDE):
    x = values[t - HIST_STEPS:t, :]      # (384, 1)
    y = values[t:t + HZN_STEPS, :]       # (96, 1)
    X_list.append(x)
    Y_list.append(y)
    meta.append((
        full_scaled.index[t - HIST_STEPS],
        full_scaled.index[t - 1],
        full_scaled.index[t],
        full_scaled.index[t + HZN_STEPS - 1],
    ))

X = np.stack(X_list)   # (N, 384, 1)
Y = np.stack(Y_list)   # (N, 96, 1)
meta_df = pd.DataFrame(meta, columns=["x_start", "x_end", "y_start", "y_end"])
print("Windowed tensors:", X.shape, Y.shape)

T, D: 35040 1
Windowed tensors: (34560, 384, 1) (34560, 96, 1)


In [ ]:
N = len(X)
n_train = int(TRAIN_FRAC * N)
n_val   = int(VAL_FRAC * N)
n_test  = N - n_train - n_val

idx_train = slice(0, n_train)
idx_val   = slice(n_train, n_train + n_val)
idx_test  = slice(n_train + n_val, N)

X_train, Y_train = X[idx_train], Y[idx_train]
X_val,   Y_val   = X[idx_val],   Y[idx_val]
X_test,  Y_test  = X[idx_test],  Y[idx_test]

print("Splits:")
print("  train:", X_train.shape)
print("  val  :", X_val.shape)
print("  test :", X_test.shape)

np.save(os.path.join(ARTIFACTS_DIR, "X_test.npy"), X_test)
np.save(os.path.join(ARTIFACTS_DIR, "Y_test.npy"), Y_test)
print("Saved full test X/Y arrays to ARTIFACTS_DIR.")

Splits:
  train: (24192, 384, 1)
  val  : (3456, 384, 1)
  test : (6912, 384, 1)
Saved full test X/Y arrays to ARTIFACTS_DIR.


In [ ]:
class ElecSeq2SeqDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.from_numpy(X)  # (N, hist, D)
        self.Y = torch.from_numpy(Y)  # (N, hzn, D)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, i):
        return self.X[i], self.Y[i]

train_dl = DataLoader(
    ElecSeq2SeqDataset(X_train, Y_train),
    batch_size=BATCH, shuffle=True,
    num_workers=2, pin_memory=True
)
val_dl = DataLoader(
    ElecSeq2SeqDataset(X_val, Y_val),
    batch_size=BATCH, shuffle=False,
    num_workers=2, pin_memory=True
)
test_dl = DataLoader(
    ElecSeq2SeqDataset(X_test, Y_test),
    batch_size=BATCH, shuffle=False,
    num_workers=2, pin_memory=True
)

In [ ]:
def patchify(x, patch):
    if patch == 1:
        return x
    B, T, D = x.shape
    T_trim = (T // patch) * patch
    x = x[:, :T_trim, :].reshape(B, T_trim // patch, patch, D).mean(dim=2)
    return x

def unpatchify(y_patch, patch, T_out):
    if patch == 1:
        return y_patch[:, :T_out, :]
    y = y_patch.repeat_interleave(patch, dim=1)
    return y[:, :T_out, :]

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=6000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class ElecSeq2SeqTransformer(nn.Module):
    def __init__(self,
                 d_feats,
                 d_model=256,
                 nhead=4,
                 num_layers=3,
                 dim_ff=512,
                 dropout=0.1,
                 hist_steps=HIST_STEPS,
                 hzn_steps=HZN_STEPS,
                 patch=PATCH):
        super().__init__()
        self.d_feats    = d_feats
        self.d_model    = d_model
        self.hist_steps = hist_steps
        self.hzn_steps  = hzn_steps
        self.patch      = patch

        self.in_proj = nn.Linear(d_feats, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.enc_pos = PositionalEncoding(d_model, max_len=(hist_steps // patch) + 16)

        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=num_layers)
        self.dec_pos = PositionalEncoding(d_model, max_len=(hzn_steps // patch) + 16)

        self.out_proj = nn.Linear(d_model, d_feats)

        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x):
        B, T, D = x.shape
        x_p = patchify(x, self.patch)
        z   = self.in_proj(x_p)
        z   = self.enc_pos(z)
        mem = self.encoder(z)

        dec_Tp = self.hzn_steps // self.patch
        tgt = torch.zeros(B, dec_Tp, self.d_model, device=x.device)
        tgt = self.dec_pos(tgt)
        out = self.decoder(tgt, mem)
        y_p = self.out_proj(out)
        y   = unpatchify(y_p, self.patch, self.hzn_steps)
        return y

In [ ]:
mse_loss = nn.MSELoss()

def rmse_only(pred, true):
    mse = mse_loss(pred, true)
    return torch.sqrt(mse + 1e-12)

def seq_rmse_np(y_true_np, y_pred_np):
    return float(np.sqrt(mean_squared_error(y_true_np.ravel(), y_pred_np.ravel())))

In [ ]:
#pgd attack

def pgd_attack(attacker_model,
               X,
               Y,
               eps,
               alpha,
               iters,
               random_start=True,
               return_grad_norm=False):

    attacker_model.eval()

    if random_start:
        noise = torch.empty_like(X).uniform_(-eps, eps)
        X_adv = (X + noise).clamp(0.0, 1.0)
    else:
        X_adv = X.clone()

    X_adv = X_adv.detach().requires_grad_(True)
    last_grad = None

    for _ in range(iters):
        preds = attacker_model(X_adv)
        loss  = mse_loss(preds, Y)

        if X_adv.grad is not None:
            X_adv.grad.zero_()

        loss.backward()
        grad = X_adv.grad
        last_grad = grad

        X_adv = (X_adv + alpha * grad.sign()).detach()
        delta = torch.clamp(X_adv - X, -eps, eps)
        X_adv = (X + delta).clamp(0.0, 1.0).detach().requires_grad_(True)

    X_adv = X_adv.detach()
    if return_grad_norm and last_grad is not None:
        gn = last_grad.view(last_grad.size(0), -1).norm(p=2, dim=1)
        return X_adv, gn.detach()
    return X_adv

In [ ]:
@torch.no_grad()
def eval_one_epoch(model, loader):
    model.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        p = model(xb)
        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())
    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

In [ ]:
def eval_under_attack(model_to_eval,
                      attacker_model,
                      loader,
                      atk_fn,
                      atk_kwargs):

    model_to_eval.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        xa = atk_fn(attacker_model, xb, yb, **atk_kwargs)

        with torch.no_grad():
            p = model_to_eval(xa)
        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())

    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

In [ ]:
def eval_pgd_with_artifacts(model,
                            loader,
                            eps,
                            model_name,
                            run_i,
                            attack_name="pgd",
                            iters=PGD_ITERS,
                            random_start=PGD_RANDOM_START_EVAL,
                            max_save_batches=1,
                            max_examples_per_batch=4):

    model.eval()

    H = HZN_STEPS
    D = 1

    alpha = eps / float(iters)

    sum_sq_clean = 0.0
    sum_sq_adv   = 0.0
    total_count  = 0

    sum_sq_clean_h = np.zeros(H, dtype=np.float64)
    sum_sq_adv_h   = np.zeros(H, dtype=np.float64)
    horizon_count  = 0

    grad_norm_rows = []

    sample_x = []
    sample_y = []
    sample_y_clean = []
    sample_y_adv   = []
    sample_x_adv   = []
    sample_delta   = []

    for batch_idx, (xb, yb) in enumerate(loader):
        xb = xb.to(device)
        yb = yb.to(device)

        xa, gn = pgd_attack(
            model,
            xb,
            yb,
            eps=eps,
            alpha=alpha,
            iters=iters,
            random_start=random_start,
            return_grad_norm=True
        )

        with torch.no_grad():
            y_clean = model(xb)
            y_adv   = model(xa)

        # errors
        err_clean = (y_clean - yb) ** 2
        err_adv   = (y_adv   - yb) ** 2

        bs = xb.size(0)
        sum_sq_clean += err_clean.sum().item()
        sum_sq_adv   += err_adv.sum().item()
        total_count  += bs * H * D

        err_clean_np = err_clean.detach().cpu().numpy()
        err_adv_np   = err_adv.detach().cpu().numpy()

        sum_sq_clean_h += err_clean_np.sum(axis=(0, 2))
        sum_sq_adv_h   += err_adv_np.sum(axis=(0, 2))
        horizon_count  += bs * D

        # gradient norms
        gn_np = gn.detach().cpu().numpy()
        grad_norm_rows.append(pd.DataFrame({
            "run": run_i,
            "attack": attack_name,
            "epsilon": eps,
            "model": model_name,
            "batch_idx": batch_idx,
            "grad_norm": gn_np,
        }))

        # sample artifacts
        if batch_idx < max_save_batches:
            n_save = min(bs, max_examples_per_batch)
            xb_np      = xb[:n_save].detach().cpu().numpy()
            yb_np      = yb[:n_save].detach().cpu().numpy()
            yc_np      = y_clean[:n_save].detach().cpu().numpy()
            ya_np      = y_adv[:n_save].detach().cpu().numpy()
            xa_np      = xa[:n_save].detach().cpu().numpy()
            delta_np   = np.abs(ya_np - yc_np)

            sample_x.append(xb_np)
            sample_y.append(yb_np)
            sample_y_clean.append(yc_np)
            sample_y_adv.append(ya_np)
            sample_x_adv.append(xa_np)
            sample_delta.append(delta_np)

    # Aggregate grad norms
    if len(grad_norm_rows) > 0:
        df_gn = pd.concat(grad_norm_rows, ignore_index=True)
        append_df_to_csv(df_gn, GRAD_NORMS_CSV)

    # Global RMSE
    rmse_clean = math.sqrt(sum_sq_clean / max(1, total_count))
    rmse_adv   = math.sqrt(sum_sq_adv   / max(1, total_count))

    # Per-horizon RMSE
    rmse_clean_h = np.sqrt(sum_sq_clean_h / max(1, horizon_count))
    rmse_adv_h   = np.sqrt(sum_sq_adv_h   / max(1, horizon_count))

    df_ph = pd.DataFrame({
        "run": run_i,
        "attack": attack_name,
        "epsilon": eps,
        "model": model_name,
        "horizon_idx": np.arange(H),
        "rmse_clean": rmse_clean_h,
        "rmse_adv": rmse_adv_h,
    })
    append_df_to_csv(df_ph, PER_HORIZON_CSV)

    # save sample arrays + plots
    if len(sample_x) > 0:
        sx  = np.concatenate(sample_x, axis=0)
        sy  = np.concatenate(sample_y, axis=0)
        syc = np.concatenate(sample_y_clean, axis=0)
        sya = np.concatenate(sample_y_adv, axis=0)
        sxa = np.concatenate(sample_x_adv, axis=0)
        sdl = np.concatenate(sample_delta, axis=0)

        prefix = f"run{run_i:02d}_{model_name}_{attack_name}_eps{eps:.2f}"
        npz_path = os.path.join(ARTIFACTS_DIR, f"{prefix}_samples.npz")
        np.savez_compressed(
            npz_path,
            X=sx, Y=sy,
            Y_clean=syc,
            Y_adv=sya,
            X_adv=sxa,
            delta=sdl
        )

        t_hzn = np.arange(H)

        # GT vs clean vs adv
        ex_y   = sy[0, :, 0]
        ex_yc  = syc[0, :, 0]
        ex_ya  = sya[0, :, 0]

        plt.figure(figsize=(8, 4))
        plt.plot(t_hzn, ex_y,  label="GT")
        plt.plot(t_hzn, ex_yc, label="Clean pred")
        plt.plot(t_hzn, ex_ya, label="Adv pred")
        plt.xlabel("Horizon step")
        plt.ylabel("Scaled load")
        plt.title(f"GT vs Clean vs Adv | {prefix}")
        plt.legend()
        gt_plot = os.path.join(PLOTS_DIR, f"{prefix}_gt_clean_adv.png")
        plt.tight_layout()
        plt.savefig(gt_plot)
        plt.close()

        # Horizon-wise RMSE
        plt.figure(figsize=(8, 4))
        plt.plot(t_hzn, rmse_clean_h, label="Clean RMSE")
        plt.plot(t_hzn, rmse_adv_h,   label="Adv RMSE")
        plt.xlabel("Horizon step")
        plt.ylabel("RMSE")
        plt.title(f"Horizon-wise RMSE | {prefix}")
        plt.legend()
        hrmse_plot = os.path.join(PLOTS_DIR, f"{prefix}_horizon_rmse.png")
        plt.tight_layout()
        plt.savefig(hrmse_plot)
        plt.close()

        # Delta histogram
        plt.figure(figsize=(6, 4))
        plt.hist(sdl.flatten(), bins=50)
        plt.xlabel("|y_adv - y_clean|")
        plt.ylabel("Count")
        plt.title(f"Attack delta distribution | {prefix}")
        delta_plot = os.path.join(PLOTS_DIR, f"{prefix}_delta_hist.png")
        plt.tight_layout()
        plt.savefig(delta_plot)
        plt.close()

        # Input perturbation heatmap
        ex_x  = sx[0, :, 0]
        ex_xa = sxa[0, :, 0]
        pert  = ex_xa - ex_x
        plt.figure(figsize=(10, 2))
        plt.imshow(pert[None, :], aspect="auto", interpolation="nearest")
        plt.colorbar(label="delta X")
        plt.xlabel("History step")
        plt.yticks([])
        plt.title(f"Input perturbation heatmap | {prefix}")
        heatmap_plot = os.path.join(PLOTS_DIR, f"{prefix}_perturb_heatmap.png")
        plt.tight_layout()
        plt.savefig(heatmap_plot)
        plt.close()

        df_idx = pd.DataFrame([{
            "run": run_i,
            "model": model_name,
            "attack": attack_name,
            "epsilon": eps,
            "samples_npz": npz_path,
            "gt_clean_adv_plot": gt_plot,
            "horizon_rmse_plot": hrmse_plot,
            "delta_hist_plot": delta_plot,
            "perturb_heatmap_plot": heatmap_plot,
        }])
        append_df_to_csv(df_idx, ARTIFACT_INDEX_CSV)

    return rmse_clean, rmse_adv

In [ ]:
if not os.path.exists(BASE_CKPT):
    raise FileNotFoundError(f"Base checkpoint not found at {BASE_CKPT}")

print("Loading base checkpoint:", BASE_CKPT)

base_cpu = ElecSeq2SeqTransformer(
    d_feats=D_FEATS,
    d_model=D_MODEL,
    nhead=N_HEAD,
    num_layers=N_LAYERS,
    dim_ff=DIM_FF,
    dropout=DROPOUT,
    hist_steps=HIST_STEPS,
    hzn_steps=HZN_STEPS,
    patch=PATCH
).to("cpu")
base_cpu.load_state_dict(torch.load(BASE_CKPT, map_location="cpu"))

best_base = ElecSeq2SeqTransformer(
    d_feats=D_FEATS,
    d_model=D_MODEL,
    nhead=N_HEAD,
    num_layers=N_LAYERS,
    dim_ff=DIM_FF,
    dropout=DROPOUT,
    hist_steps=HIST_STEPS,
    hzn_steps=HZN_STEPS,
    patch=PATCH
).to(device)
best_base.load_state_dict(torch.load(BASE_CKPT, map_location=device))

# log base model size
n_params = model_n_params(base_cpu)
size_mb  = model_file_size_mb(BASE_CKPT)
df_ms = pd.DataFrame([{
    "model_name": "base_transformer_pgd",
    "path": BASE_CKPT,
    "params": n_params,
    "size_mb": size_mb,
}])
append_df_to_csv(df_ms, MODEL_SIZES_CSV)

print("Base model loaded. Params:", n_params, " Size (MB):", size_mb)

Loading base checkpoint: /content/drive/MyDrive/298/Elec/SanityCheck/models/baseModel/elec_base_best.pth


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Base model loaded. Params: 3954433  Size (MB): 15.267773628234863


In [ ]:
num_students = 4
student_paths = []

print("\ncreating students")
for i in range(1, num_students + 1):
    st = copy.deepcopy(base_cpu)
    with torch.no_grad():
        for _, p in st.named_parameters():
            p.add_(1e-3 * torch.randn_like(p))
    outp = os.path.join(STUDENTS_DIR, f"student_{i:02d}.pth")
    torch.save(st.state_dict(), outp)
    student_paths.append(outp)
    print("Saved student:", outp)

student_paths = sorted(student_paths)
print("Students:", student_paths)

# log student model sizes
for sp in student_paths:
    size_mb = model_file_size_mb(sp)
    df_ms = pd.DataFrame([{
        "model_name": os.path.basename(sp).replace(".pth", ""),
        "path": sp,
        "params": model_n_params(base_cpu),
        "size_mb": size_mb,
    }])
    append_df_to_csv(df_ms, MODEL_SIZES_CSV)


creating students
Saved student: /content/drive/MyDrive/298/Elec/SanityCheck/pgd/students/student_01.pth
Saved student: /content/drive/MyDrive/298/Elec/SanityCheck/pgd/students/student_02.pth
Saved student: /content/drive/MyDrive/298/Elec/SanityCheck/pgd/students/student_03.pth
Saved student: /content/drive/MyDrive/298/Elec/SanityCheck/pgd/students/student_04.pth
Students: ['/content/drive/MyDrive/298/Elec/SanityCheck/pgd/students/student_01.pth', '/content/drive/MyDrive/298/Elec/SanityCheck/pgd/students/student_02.pth', '/content/drive/MyDrive/298/Elec/SanityCheck/pgd/students/student_03.pth', '/content/drive/MyDrive/298/Elec/SanityCheck/pgd/students/student_04.pth']


In [ ]:
adv_student_paths = []

print("\npgd adversarial training of students")

for sp in student_paths:
    idx = os.path.basename(sp).split("_")[1].split(".")[0]
    print(f"\n[Student {idx}] PGD adversarial training...")

    stu = ElecSeq2SeqTransformer(
        d_feats=D_FEATS,
        d_model=D_MODEL,
        nhead=N_HEAD,
        num_layers=N_LAYERS,
        dim_ff=DIM_FF,
        dropout=DROPOUT,
        hist_steps=HIST_STEPS,
        hzn_steps=HZN_STEPS,
        patch=PATCH
    ).to(device)
    stu.load_state_dict(torch.load(sp, map_location=device))

    opt = optim.AdamW(stu.parameters(), lr=LR_STUDENT)

    stu_history = []

    hb_flag, hb_thread = start_heartbeat(f"student_{idx}_pgd_train")
    start_time = time.time()

    try:
        for ep in range(1, EPOCHS_STU + 1):
            stu.train()
            tot = cnt = 0
            for xb, yb in train_dl:
                xb = xb.to(device)
                yb = yb.to(device)

                # clean prediction
                pred_c = stu(xb)
                Lc = rmse_only(pred_c, yb)

                # multi epsilon PGD adv training
                adv_losses = []
                for eps in PGD_TRAIN_EPS_LIST:
                    alpha = eps / float(PGD_ITERS)
                    xa = pgd_attack(
                        stu, xb, yb,
                        eps=eps,
                        alpha=alpha,
                        iters=PGD_ITERS,
                        random_start=PGD_RANDOM_START_TRAIN
                    )
                    pred_a = stu(xa)
                    La = rmse_only(pred_a, yb)
                    adv_losses.append(La)

                if len(adv_losses) > 0:
                    La_mean = torch.stack(adv_losses).mean()
                else:
                    La_mean = 0.0 * Lc

                loss = 0.25 * Lc + 0.75 * La_mean

                opt.zero_grad()
                loss.backward()
                opt.step()

                bs = xb.size(0)
                tot += loss.item() * bs
                cnt += bs

            avg_loss = tot / max(1, cnt)
            elapsed  = (time.time() - start_time) / 60.0
            print(f"  [Student {idx}] Epoch {ep:02d} | avg_loss={avg_loss:.4f} | elapsed={elapsed:.2f} min")

            stu_history.append({
                "student_idx": int(idx),
                "epoch": ep,
                "avg_loss": avg_loss,
                "elapsed_min": elapsed,
            })

    except Exception as e:
        # create log on crash and keep a checkpoint
        stop_heartbeat(hb_flag, hb_thread)
        msg = f"[Student {idx}] PGD adv training crash: {repr(e)}\n{traceback.format_exc()}"
        print(msg)
        log_error(msg)
        ckpt_crash = os.path.join(STUDENTS_ADV_DIR, f"student_{idx}_pgd_CRASHED.pth")
        torch.save(stu.to("cpu").state_dict(), ckpt_crash)
        raise

    finally:
        stop_heartbeat(hb_flag, hb_thread)

    # save per student training history csv
    stu_hist_path = os.path.join(RESULTS_DIR, f"student_{idx}_pgd_train_history.csv")
    df_stu_hist = pd.DataFrame(stu_history)
    df_stu_hist.to_csv(stu_hist_path, index=False)
    print(f"  [Student {idx}] Saved PGD training history to:", stu_hist_path)

    outp = os.path.join(STUDENTS_ADV_DIR, f"student_{idx}_pgd_adv.pth")
    torch.save(stu.to("cpu").state_dict(), outp)
    adv_student_paths.append(outp)
    print(f"  [Student {idx}] Saved PGD adv student:", outp)

    # log adv student model size
    size_mb = model_file_size_mb(outp)
    df_ms = pd.DataFrame([{
        "model_name": f"student_{idx}_pgd_adv",
        "path": outp,
        "params": model_n_params(base_cpu),
        "size_mb": size_mb,
    }])
    append_df_to_csv(df_ms, MODEL_SIZES_CSV)

adv_student_paths = sorted(adv_student_paths)
print("\nPGD adv students:", adv_student_paths)


pgd adversarial training of students

[Student 01] PGD adversarial training...
[heartbeat:student_01_pgd_train] running... elapsed=10.00 min
[heartbeat:student_01_pgd_train] running... elapsed=20.00 min
  [Student 01] Epoch 01 | avg_loss=0.1445 | elapsed=23.88 min
[heartbeat:student_01_pgd_train] running... elapsed=30.00 min
[heartbeat:student_01_pgd_train] running... elapsed=40.00 min
  [Student 01] Epoch 02 | avg_loss=0.1411 | elapsed=47.94 min
[heartbeat:student_01_pgd_train] running... elapsed=50.00 min
[heartbeat:student_01_pgd_train] running... elapsed=60.00 min
[heartbeat:student_01_pgd_train] running... elapsed=70.00 min
  [Student 01] Epoch 03 | avg_loss=0.1384 | elapsed=71.63 min
[heartbeat:student_01_pgd_train] running... elapsed=80.00 min
[heartbeat:student_01_pgd_train] running... elapsed=90.00 min
  [Student 01] Epoch 04 | avg_loss=0.1378 | elapsed=94.47 min
[heartbeat:student_01_pgd_train] running... elapsed=100.00 min
[heartbeat:student_01_pgd_train] running... elapsed

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


[heartbeat:student_02_pgd_train] running... elapsed=10.00 min
[heartbeat:student_02_pgd_train] running... elapsed=20.00 min
  [Student 02] Epoch 01 | avg_loss=0.1444 | elapsed=23.12 min
[heartbeat:student_02_pgd_train] running... elapsed=30.00 min
[heartbeat:student_02_pgd_train] running... elapsed=40.00 min
  [Student 02] Epoch 02 | avg_loss=0.1410 | elapsed=46.02 min
[heartbeat:student_02_pgd_train] running... elapsed=50.00 min
[heartbeat:student_02_pgd_train] running... elapsed=60.00 min
  [Student 02] Epoch 03 | avg_loss=0.1389 | elapsed=68.97 min
[heartbeat:student_02_pgd_train] running... elapsed=70.00 min
[heartbeat:student_02_pgd_train] running... elapsed=80.00 min
[heartbeat:student_02_pgd_train] running... elapsed=90.00 min
  [Student 02] Epoch 04 | avg_loss=0.1379 | elapsed=92.04 min
[heartbeat:student_02_pgd_train] running... elapsed=100.00 min
[heartbeat:student_02_pgd_train] running... elapsed=110.00 min
  [Student 02] Epoch 05 | avg_loss=0.1374 | elapsed=115.04 min
  [St

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, math, copy, random, time, threading, gc, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
DATA_PATH = "/content/drive/MyDrive/298/Elec/Data/LD2011_2014.txt"

SANITY_PGD_DIR          = "/content/drive/MyDrive/298/Elec/SanityCheck/pgd"
BASE_MODEL_DIR          = "/content/drive/MyDrive/298/Elec/SanityCheck/models/baseModel"
SANITY_RESULTS_DIR      = os.path.join(SANITY_PGD_DIR, "results")
SANITY_ARTIFACTS_DIR    = os.path.join(SANITY_RESULTS_DIR, "artifacts")
SANITY_STUDENTS_ADV_DIR = os.path.join(SANITY_PGD_DIR, "students_adv")

FINAL_PROJECT_DIR    = "/content/drive/MyDrive/298/Elec/FinalPipeline/pgd"
FINAL_RESULTS_DIR    = os.path.join(FINAL_PROJECT_DIR, "results")
FINAL_PLOTS_DIR      = os.path.join(FINAL_RESULTS_DIR, "plots")
FINAL_ARTIFACTS_DIR  = os.path.join(FINAL_RESULTS_DIR, "artifacts")

for d in [FINAL_PROJECT_DIR, FINAL_RESULTS_DIR, FINAL_PLOTS_DIR, FINAL_ARTIFACTS_DIR]:
    os.makedirs(d, exist_ok=True)

PGD_RUNS_CSV             = os.path.join(FINAL_RESULTS_DIR, "pgd_morphence_runs.csv")
PGD_STATS_CSV            = os.path.join(FINAL_RESULTS_DIR, "pgd_morphence_stats.csv")
PGD_SINGLE_RUN_CSV       = os.path.join(FINAL_RESULTS_DIR, "pgd_morphence_single_run.csv")
PGD_XFER_MATRIX_CSV      = os.path.join(FINAL_RESULTS_DIR, "pgd_transfer_matrix.csv")
PGD_WEIGHT_DIVERSITY_CSV = os.path.join(FINAL_RESULTS_DIR, "pgd_weight_diversity.csv")
PGD_PRED_DIVERSITY_CSV   = os.path.join(FINAL_RESULTS_DIR, "pgd_prediction_diversity.csv")

print("SanityCheck PGD dir:", SANITY_PGD_DIR)
print("SanityCheck PGD artifacts dir:", SANITY_ARTIFACTS_DIR)
print("SanityCheck PGD students_adv dir:", SANITY_STUDENTS_ADV_DIR)
print("FinalPipeline PGD results dir:", FINAL_RESULTS_DIR)

SanityCheck PGD dir: /content/drive/MyDrive/298/Elec/SanityCheck/pgd
SanityCheck PGD artifacts dir: /content/drive/MyDrive/298/Elec/SanityCheck/pgd/results/artifacts
SanityCheck PGD students_adv dir: /content/drive/MyDrive/298/Elec/SanityCheck/pgd/students_adv
FinalPipeline PGD results dir: /content/drive/MyDrive/298/Elec/FinalPipeline/pgd/results


In [ ]:
FREQ = "15min"
STEPS_PER_HOUR = 4
HOURS_PER_DAY  = 24

HIST_HOURS  = 96
HZN_HOURS   = 24
HIST_STEPS  = HIST_HOURS * STEPS_PER_HOUR   # 384
HZN_STEPS   = HZN_HOURS  * STEPS_PER_HOUR   # 96

BATCH       = 16
PATCH       = 4
D_FEATS     = 1
D_MODEL     = 256
N_HEAD      = 4
N_LAYERS    = 3
DIM_FF      = 512
DROPOUT     = 0.1

PGD_EVAL_EPS_LIST     = [0.10, 0.20, 0.30, 0.50]
PGD_ITERS             = 10
PGD_RANDOM_START_EVAL = True

N_RUNS = 30
HEARTBEAT_SECS = 600  # 10 minutes

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", device)

Device: cuda


In [ ]:
def start_heartbeat(name="task"):
    flag = {"on": True}
    start_time = time.time()

    def _hb():
        while flag["on"]:
            time.sleep(HEARTBEAT_SECS)
            if flag["on"]:
                elapsed = (time.time() - start_time) / 60.0
                print(f"[heartbeat:{name}] running... elapsed={elapsed:.2f} min", flush=True)

    t = threading.Thread(target=_hb, daemon=True)
    t.start()
    return flag, t

def stop_heartbeat(flag, thread):
    flag["on"] = False
    try:
        thread.join(timeout=1)
    except Exception:
        pass

def seq_rmse_np(y_true_np, y_pred_np):
    return float(np.sqrt(mean_squared_error(y_true_np.ravel(), y_pred_np.ravel())))

In [ ]:
X_test_path = os.path.join(SANITY_ARTIFACTS_DIR, "X_test.npy")
Y_test_path = os.path.join(SANITY_ARTIFACTS_DIR, "Y_test.npy")

assert os.path.exists(X_test_path), f"Missing {X_test_path}"
assert os.path.exists(Y_test_path), f"Missing {Y_test_path}"

X_test = np.load(X_test_path)
Y_test = np.load(Y_test_path)

print("Loaded X_test:", X_test.shape)
print("Loaded Y_test:", Y_test.shape)

class ElecSeq2SeqDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.from_numpy(X)  # (N, hist, D)
        self.Y = torch.from_numpy(Y)  # (N, hzn, D)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, i):
        return self.X[i], self.Y[i]

test_dl = DataLoader(
    ElecSeq2SeqDataset(X_test, Y_test),
    batch_size=BATCH,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

Loaded X_test: (6912, 384, 1)
Loaded Y_test: (6912, 96, 1)


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=6000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

In [ ]:
def patchify(x, patch):
    if patch == 1:
        return x
    B, T, D = x.shape
    T_trim = (T // patch) * patch
    x = x[:, :T_trim, :].reshape(B, T_trim // patch, patch, D).mean(dim=2)
    return x

def unpatchify(y_patch, patch, T_out):
    if patch == 1:
        return y_patch[:, :T_out, :]
    y = y_patch.repeat_interleave(patch, dim=1)
    return y[:, :T_out, :]

In [ ]:
class ElecSeq2SeqTransformer(nn.Module):
    def __init__(self,
                 d_feats,
                 d_model=256,
                 nhead=4,
                 num_layers=3,
                 dim_ff=512,
                 dropout=0.1,
                 hist_steps=HIST_STEPS,
                 hzn_steps=HZN_STEPS,
                 patch=PATCH):
        super().__init__()
        self.d_feats    = d_feats
        self.d_model    = d_model
        self.hist_steps = hist_steps
        self.hzn_steps  = hzn_steps
        self.patch      = patch

        self.in_proj = nn.Linear(d_feats, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.enc_pos = PositionalEncoding(d_model, max_len=(hist_steps // patch) + 16)

        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=num_layers)
        self.dec_pos = PositionalEncoding(d_model, max_len=(hzn_steps // patch) + 16)

        self.out_proj = nn.Linear(d_model, d_feats)

        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x):
        B, T, D = x.shape
        x_p = patchify(x, self.patch)
        z   = self.in_proj(x_p)
        z   = self.enc_pos(z)
        mem = self.encoder(z)

        dec_Tp = self.hzn_steps // self.patch
        tgt = torch.zeros(B, dec_Tp, self.d_model, device=x.device)
        tgt = self.dec_pos(tgt)
        out = self.decoder(tgt, mem)
        y_p = self.out_proj(out)
        y   = unpatchify(y_p, self.patch, self.hzn_steps)
        return y

In [ ]:
mse_loss = nn.MSELoss()

def pgd_attack(attacker_model,
               X,
               Y,
               eps,
               alpha,
               iters,
               random_start=True):

    attacker_model.eval()

    if random_start:
        noise = torch.empty_like(X).uniform_(-eps, eps)
        X_adv = (X + noise).clamp(0.0, 1.0)
    else:
        X_adv = X.clone()

    X_adv = X_adv.detach().requires_grad_(True)

    for _ in range(iters):
        preds = attacker_model(X_adv)
        loss  = mse_loss(preds, Y)

        if X_adv.grad is not None:
            X_adv.grad.zero_()

        loss.backward()
        grad = X_adv.grad

        X_adv = (X_adv + alpha * grad.sign()).detach()
        delta = torch.clamp(X_adv - X, -eps, eps)
        X_adv = (X + delta).clamp(0.0, 1.0).detach().requires_grad_(True)

    return X_adv.detach()

In [ ]:
base_ckpt = os.path.join(BASE_MODEL_DIR, "elec_base_best.pth")
assert os.path.exists(base_ckpt), f"Missing base checkpoint: {base_ckpt}"

best_base = ElecSeq2SeqTransformer(
    d_feats=D_FEATS,
    d_model=D_MODEL,
    nhead=N_HEAD,
    num_layers=N_LAYERS,
    dim_ff=DIM_FF,
    dropout=DROPOUT,
    hist_steps=HIST_STEPS,
    hzn_steps=HZN_STEPS,
    patch=PATCH
).to(device)
best_base.load_state_dict(torch.load(base_ckpt, map_location=device))
best_base.eval()
print("Loaded base model from:", base_ckpt)

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Loaded base model from: /content/drive/MyDrive/298/Elec/SanityCheck/models/baseModel/elec_base_best.pth


In [ ]:
pgd_student_paths = sorted(
    [
        os.path.join(SANITY_STUDENTS_ADV_DIR, f)
        for f in os.listdir(SANITY_STUDENTS_ADV_DIR)
        if f.endswith("_pgd_adv.pth")
    ]
)

assert len(pgd_student_paths) > 0, "No *_pgd_adv.pth found in PGD students_adv dir!"
print("\nPGD adv students (from SanityCheck/pgd):")
for p in pgd_student_paths:
    print("  ", p)

pgd_adv_students = []
for sp in pgd_student_paths:
    stu = ElecSeq2SeqTransformer(
        d_feats=D_FEATS,
        d_model=D_MODEL,
        nhead=N_HEAD,
        num_layers=N_LAYERS,
        dim_ff=DIM_FF,
        dropout=DROPOUT,
        hist_steps=HIST_STEPS,
        hzn_steps=HZN_STEPS,
        patch=PATCH
    ).to(device)
    stu.load_state_dict(torch.load(sp, map_location=device))
    stu.eval()
    pgd_adv_students.append(stu)

pgd_student_names = [os.path.basename(p).replace(".pth", "") for p in pgd_student_paths]
print(f"Loaded {len(pgd_adv_students)} PGD adversarially trained students.")


PGD adv students (from SanityCheck/pgd):
   /content/drive/MyDrive/298/Elec/SanityCheck/pgd/students_adv/student_01_pgd_adv.pth
   /content/drive/MyDrive/298/Elec/SanityCheck/pgd/students_adv/student_02_pgd_adv.pth
   /content/drive/MyDrive/298/Elec/SanityCheck/pgd/students_adv/student_03_pgd_adv.pth
   /content/drive/MyDrive/298/Elec/SanityCheck/pgd/students_adv/student_04_pgd_adv.pth
Loaded 4 PGD adversarially trained students.


In [ ]:
@torch.no_grad()
def eval_clean_rmse(model, loader):
    model.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        p  = model(xb)
        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())
    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

@torch.no_grad()
def eval_ensemble_rmse(models, loader):
    for m in models:
        m.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        preds = []
        for m in models:
            preds.append(m(xb))
        preds = torch.stack(preds, dim=0).mean(dim=0)  # (B,H,D)
        ys.append(yb.cpu().numpy())
        ps.append(preds.cpu().numpy())
    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

def eval_pair_pgd(defender, attacker, loader, eps,
                  iters=PGD_ITERS,
                  random_start=PGD_RANDOM_START_EVAL):
    defender.eval()
    attacker.eval()
    alpha = eps / float(iters)

    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        xa = pgd_attack(
            attacker,
            xb,
            yb,
            eps=eps,
            alpha=alpha,
            iters=iters,
            random_start=random_start,
        )
        with torch.no_grad():
            p = defender(xa)

        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())

    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

def eval_random_switch_pgd(models, loader, eps,
                           iters=PGD_ITERS,
                           random_start=PGD_RANDOM_START_EVAL):
    for m in models:
        m.eval()

    ys, ps = [], []
    n_stu = len(models)
    alpha = eps / float(iters)

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        atk_idx  = random.randrange(n_stu)
        def_idx  = random.randrange(n_stu)
        attacker = models[atk_idx]
        defender = models[def_idx]

        xa = pgd_attack(
            attacker,
            xb,
            yb,
            eps=eps,
            alpha=alpha,
            iters=iters,
            random_start=random_start,
        )
        with torch.no_grad():
            p = defender(xa)

        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())

    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

In [ ]:
def flatten_params(model):
    return torch.cat([p.detach().cpu().view(-1) for p in model.parameters()])

def compute_weight_diversity_pgd(students, names):
    vecs = [flatten_params(m) for m in students]
    k = len(vecs)
    rows = []
    for i, j in itertools.combinations(range(k), 2):
        d = torch.norm(vecs[i] - vecs[j], p=2).item()
        rows.append({
            "student_i": names[i],
            "student_j": names[j],
            "l2_distance": d,
        })
    df_w = pd.DataFrame(rows)
    df_w.to_csv(PGD_WEIGHT_DIVERSITY_CSV, index=False)
    print("Saved PGD weight diversity to:", PGD_WEIGHT_DIVERSITY_CSV)
    return df_w

@torch.no_grad()
def compute_prediction_diversity_pgd(students, loader):
    for m in students:
        m.eval()

    k = len(students)
    H = HZN_STEPS

    sum_pred  = np.zeros(H, dtype=np.float64)
    sum_pred2 = np.zeros(H, dtype=np.float64)
    count     = 0

    for xb, yb in loader:
        xb = xb.to(device)
        preds_stu = []
        for m in students:
            preds_stu.append(m(xb).cpu().numpy())  # (B,H,1)
        preds_stu = np.stack(preds_stu, axis=0)  # (k,B,H,1)

        preds_mean_batch = preds_stu.mean(axis=1).squeeze(-1)  # (k,H)

        sum_pred  += preds_mean_batch.mean(axis=0)
        sum_pred2 += (preds_mean_batch ** 2).mean(axis=0)
        count     += 1

    mean_pred  = sum_pred / max(1, count)
    mean_pred2 = sum_pred2 / max(1, count)
    var_pred   = np.maximum(mean_pred2 - mean_pred**2, 0.0)

    df_p = pd.DataFrame({
        "horizon_idx": np.arange(H),
        "mean_pred": mean_pred,
        "var_pred": var_pred,
    })
    df_p.to_csv(PGD_PRED_DIVERSITY_CSV, index=False)
    print("Saved PGD prediction diversity to:", PGD_PRED_DIVERSITY_CSV)
    return df_p

print("\nComputing PGD diversity metrics (once)...")
_ = compute_weight_diversity_pgd(pgd_adv_students, pgd_student_names)
_ = compute_prediction_diversity_pgd(pgd_adv_students, test_dl)



Computing PGD diversity metrics (once)...
Saved PGD weight diversity to: /content/drive/MyDrive/298/Elec/FinalPipeline/pgd/results/pgd_weight_diversity.csv
Saved PGD prediction diversity to: /content/drive/MyDrive/298/Elec/FinalPipeline/pgd/results/pgd_prediction_diversity.csv


In [ ]:
pgd_xfer_rows = []
for eps in PGD_EVAL_EPS_LIST:
    print(f"\nComputing PGD transfer matrix for eps={eps}...")
    for i, atk_name in enumerate(pgd_student_names):
        for j, def_name in enumerate(pgd_student_names):
            atk_model = pgd_adv_students[i]
            def_model = pgd_adv_students[j]
            rmse_ij = eval_pair_pgd(def_model, atk_model, test_dl, eps=eps)
            pgd_xfer_rows.append({
                "epsilon": eps,
                "attacker": atk_name,
                "defender": def_name,
                "rmse_adv": rmse_ij,
            })
            print(f"  atk={atk_name} -> def={def_name} | eps={eps:.2f} | RMSE={rmse_ij:.5f}")

df_pgd_xfer = pd.DataFrame(pgd_xfer_rows)
df_pgd_xfer.to_csv(PGD_XFER_MATRIX_CSV, index=False)
print("\nSaved PGD transfer matrix to:", PGD_XFER_MATRIX_CSV)


Computing PGD transfer matrix for eps=0.1...
  atk=student_01_pgd_adv -> def=student_01_pgd_adv | eps=0.10 | RMSE=0.18666
  atk=student_01_pgd_adv -> def=student_02_pgd_adv | eps=0.10 | RMSE=0.18771
  atk=student_01_pgd_adv -> def=student_03_pgd_adv | eps=0.10 | RMSE=0.19586
  atk=student_01_pgd_adv -> def=student_04_pgd_adv | eps=0.10 | RMSE=0.20071
  atk=student_02_pgd_adv -> def=student_01_pgd_adv | eps=0.10 | RMSE=0.18650
  atk=student_02_pgd_adv -> def=student_02_pgd_adv | eps=0.10 | RMSE=0.18800
  atk=student_02_pgd_adv -> def=student_03_pgd_adv | eps=0.10 | RMSE=0.19585
  atk=student_02_pgd_adv -> def=student_04_pgd_adv | eps=0.10 | RMSE=0.20073
  atk=student_03_pgd_adv -> def=student_01_pgd_adv | eps=0.10 | RMSE=0.18619
  atk=student_03_pgd_adv -> def=student_02_pgd_adv | eps=0.10 | RMSE=0.18739
  atk=student_03_pgd_adv -> def=student_03_pgd_adv | eps=0.10 | RMSE=0.19570
  atk=student_03_pgd_adv -> def=student_04_pgd_adv | eps=0.10 | RMSE=0.20051
  atk=student_04_pgd_adv -> de

In [ ]:
print("\nSingle run PGD eval: base vs ensemble")
print("PGD epsilons:", PGD_EVAL_EPS_LIST)

pgd_single_seed = 3344
random.seed(pgd_single_seed)
np.random.seed(pgd_single_seed)
torch.manual_seed(pgd_single_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(pgd_single_seed)

pgd_single_rows = []   # ["attack", "epsilon", "model", "RMSE"]

# clean performance
pgd_base_clean_single   = eval_clean_rmse(best_base,  test_dl)
pgd_morph_clean_single  = eval_ensemble_rmse(pgd_adv_students, test_dl)

pgd_single_rows.append(["clean", None, "base",           pgd_base_clean_single])
pgd_single_rows.append(["clean", None, "morph_ensemble", pgd_morph_clean_single])

print(
    f"[PGD Single-run] clean | base={pgd_base_clean_single:.5f} | "
    f"morph_ensemble={pgd_morph_clean_single:.5f}",
    flush=True
)

for eps in PGD_EVAL_EPS_LIST:
    pgd_base_adv_single = eval_pair_pgd(
        defender=best_base,
        attacker=best_base,
        loader=test_dl,
        eps=eps
    )
    pgd_morph_rand_single = eval_random_switch_pgd(
        models=pgd_adv_students,
        loader=test_dl,
        eps=eps
    )

    pgd_single_rows.append(["pgd", eps, "base",       pgd_base_adv_single])
    pgd_single_rows.append(["pgd", eps, "morph_rand", pgd_morph_rand_single])

    print(
        f"[PGD Single run] eps={eps:.2f} | base_adv={pgd_base_adv_single:.5f} | "
        f"morph_rand={pgd_morph_rand_single:.5f}",
        flush=True
    )

df_pgd_single = pd.DataFrame(
    pgd_single_rows,
    columns=["attack", "epsilon", "model", "RMSE"]
)
df_pgd_single.to_csv(PGD_SINGLE_RUN_CSV, index=False)
print("\nSaved single run PGD eval CSV to:", PGD_SINGLE_RUN_CSV)


Single run PGD eval: base vs ensemble
PGD epsilons: [0.1, 0.2, 0.3, 0.5]
[PGD Single-run] clean | base=0.11506 | morph_ensemble=0.19134
[PGD Single run] eps=0.10 | base_adv=0.24941 | morph_rand=0.19304
[PGD Single run] eps=0.20 | base_adv=0.33839 | morph_rand=0.19477
[PGD Single run] eps=0.30 | base_adv=0.36821 | morph_rand=0.19934
[PGD Single run] eps=0.50 | base_adv=0.39604 | morph_rand=0.21666

Saved single run PGD eval CSV to: /content/drive/MyDrive/298/Elec/FinalPipeline/pgd/results/pgd_morphence_single_run.csv


In [ ]:
print("\n30 run PGD eval: base vs ensemble")
print("PGD epsilons:", PGD_EVAL_EPS_LIST)
print("Runs:", N_RUNS)

pgd_all_rows = []   # ["attack", "epsilon", "model", "RMSE", "run"]

pgd_start_all = time.time()
pgd_hb_flag, pgd_hb_thread = start_heartbeat("eval_pgd_morphence")

try:
    for run_i in range(1, N_RUNS + 1):
        run_start = time.time()
        seed_i = 8000 + run_i
        random.seed(seed_i)
        np.random.seed(seed_i)
        torch.manual_seed(seed_i)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed_i)

        # clean
        base_clean  = eval_clean_rmse(best_base, test_dl)
        morph_clean = eval_ensemble_rmse(pgd_adv_students, test_dl)

        pgd_all_rows.append(["clean", None, "base",           base_clean,  run_i])
        pgd_all_rows.append(["clean", None, "morph_ensemble", morph_clean, run_i])

        print(f"[PGD Run {run_i:02d}] clean | base={base_clean:.5f} | morph_ens={morph_clean:.5f}",
              flush=True)

        # PGD robustness
        # white-box base, random-switch Morphence
        for eps in PGD_EVAL_EPS_LIST:
            base_adv   = eval_pair_pgd(best_base, best_base, test_dl, eps=eps)
            morph_rand = eval_random_switch_pgd(pgd_adv_students, test_dl, eps=eps)

            pgd_all_rows.append(["pgd", eps, "base",       base_adv,   run_i])
            pgd_all_rows.append(["pgd", eps, "morph_rand", morph_rand, run_i])

            print(
                f"[PGD Run {run_i:02d}] eps={eps:.2f} | base_adv={base_adv:.5f} | "
                f"morph_rand={morph_rand:.5f}",
                flush=True
            )

        # autosave after each run
        df_pgd_runs = pd.DataFrame(
            pgd_all_rows,
            columns=["attack", "epsilon", "model", "RMSE", "run"]
        )
        df_pgd_runs.to_csv(PGD_RUNS_CSV, index=False)

        pgd_stats = (
            df_pgd_runs
            .groupby(["attack", "epsilon", "model"])["RMSE"]
            .agg(mean="mean", std="std", n="count")
            .reset_index()
            .sort_values(["attack", "epsilon", "model"])
        )
        pgd_stats.to_csv(PGD_STATS_CSV, index=False)

        run_mins   = (time.time() - run_start) / 60.0
        total_mins = (time.time() - pgd_start_all) / 60.0
        print(
            f"[PGD] Completed run {run_i}/{N_RUNS} | "
            f"run_time={run_mins:.2f} min | total_elapsed={total_mins:.2f} min",
            flush=True
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        gc.collect()

finally:
    stop_heartbeat(pgd_hb_flag, pgd_hb_thread)

print("\nSaved PGD ensemble eval runs CSV to:", PGD_RUNS_CSV)
print("Saved PGD ensemble eval stats CSV to:", PGD_STATS_CSV)
print("PGD weight diversity CSV:", PGD_WEIGHT_DIVERSITY_CSV)
print("PGD prediction diversity CSV:", PGD_PRED_DIVERSITY_CSV)
print("PGD transfer matrix CSV:", PGD_XFER_MATRIX_CSV)
print("FinalPipeline PGD plots dir:", FINAL_PLOTS_DIR)
print("FinalPipeline PGD artifacts dir:", FINAL_ARTIFACTS_DIR)


30 run PGD eval: base vs ensemble
PGD epsilons: [0.1, 0.2, 0.3, 0.5]
Runs: 30
[PGD Run 01] clean | base=0.11506 | morph_ens=0.19134
[PGD Run 01] eps=0.10 | base_adv=0.24914 | morph_rand=0.19325
[PGD Run 01] eps=0.20 | base_adv=0.33879 | morph_rand=0.19350
[PGD Run 01] eps=0.30 | base_adv=0.36850 | morph_rand=0.19807
[heartbeat:eval_pgd_morphence] running... elapsed=10.00 min
[PGD Run 01] eps=0.50 | base_adv=0.39590 | morph_rand=0.21752
[PGD] Completed run 1/30 | run_time=11.23 min | total_elapsed=11.23 min
[PGD Run 02] clean | base=0.11506 | morph_ens=0.19134
[PGD Run 02] eps=0.10 | base_adv=0.24842 | morph_rand=0.19289
[PGD Run 02] eps=0.20 | base_adv=0.33817 | morph_rand=0.19363
[PGD Run 02] eps=0.30 | base_adv=0.36769 | morph_rand=0.19889
[heartbeat:eval_pgd_morphence] running... elapsed=20.00 min
[PGD Run 02] eps=0.50 | base_adv=0.39542 | morph_rand=0.21719
[PGD] Completed run 2/30 | run_time=11.20 min | total_elapsed=22.44 min
[PGD Run 03] clean | base=0.11506 | morph_ens=0.19134

In [ ]:
#stats

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np


ELEC_BASE_DIR    = "/content/drive/MyDrive/298/Elec/FinalPipeline"
PGD_RESULTS_DIR  = os.path.join(ELEC_BASE_DIR, "pgd", "results")

PGD_RUNS_CSV = os.path.join(PGD_RESULTS_DIR, "pgd_morphence_runs.csv")

print("PGD runs CSV:", PGD_RUNS_CSV)

os.makedirs(PGD_RESULTS_DIR, exist_ok=True)

# compute stats for PGD
def compute_stats_pgd(
    runs_csv=PGD_RUNS_CSV,
    results_dir=PGD_RESULTS_DIR,
    prefix="pgd",
    attack_key="pgd",
):
    """
    Expects a runs CSV with columns:
      attack, epsilon, model, RMSE, run
    and 'attack' values: 'clean' + attack_key (e.g., 'pgd').
    """
    print("\n==============================================")
    print("Processing PGD runs")
    print(f"Runs CSV: {runs_csv}")
    print("==============================================")

    if not os.path.exists(runs_csv):
        print(f"[WARN] Runs CSV not found for PGD: {runs_csv}")
        return

    df_runs = pd.read_csv(runs_csv)
    print("Loaded runs:", df_runs.shape)
    print(df_runs.head())

    # clean stats (base vs ensemble)
    df_clean = df_runs[df_runs["attack"] == "clean"].copy()

    df_clean["model"] = pd.Categorical(
        df_clean["model"],
        categories=["base", "morph_ensemble"],
        ordered=True,
    )

    clean_stats = (
        df_clean
        .groupby("model", observed=False)["RMSE"]
        .agg(
            mean   ="mean",
            std    ="std",
            min    ="min",
            q1     =lambda s: s.quantile(0.25),
            median ="median",
            q3     =lambda s: s.quantile(0.75),
            max    ="max",
            n      ="count",
        )
        .reset_index()
        .sort_values("model")
    )

    print("\n[PGD | Clean] stats:")
    print(clean_stats.to_string(index=False))

    clean_csv = os.path.join(results_dir, f"{prefix}_clean_stats.csv")
    clean_stats.to_csv(clean_csv, index=False)
    print("Saved clean stats to:", clean_csv)

    # PGD stats (base vs morph_rand)
    df_pgd = df_runs[df_runs["attack"] == attack_key].copy()

    def model_pgd_stats(df, model_name):
        st = (
            df[df["model"] == model_name]
            .groupby("epsilon", observed=False)["RMSE"]
            .agg(
                mean   ="mean",
                std    ="std",
                min    ="min",
                q1     =lambda s: s.quantile(0.25),
                median ="median",
                q3     =lambda s: s.quantile(0.75),
                max    ="max",
                n      ="count",
            )
            .reset_index()
            .sort_values("epsilon")
        )

        print(f"\n[PGD | {attack_key}] {model_name} stats:")
        print(st.to_string(index=False))

        out_csv = os.path.join(
            results_dir, f"{prefix}_{attack_key}_{model_name}_stats.csv"
        )
        st.to_csv(out_csv, index=False)
        print(f"Saved:", out_csv)

    # base white-box stats under PGD
    model_pgd_stats(df_pgd, "base")

    # Morphence: random-switch ensemble under PGD
    model_pgd_stats(df_pgd, "morph_rand")


# run PGD stats
compute_stats_pgd()


Mounted at /content/drive
PGD runs CSV: /content/drive/MyDrive/298/Elec/FinalPipeline/pgd/results/pgd_morphence_runs.csv

Processing PGD runs
Runs CSV: /content/drive/MyDrive/298/Elec/FinalPipeline/pgd/results/pgd_morphence_runs.csv
Loaded runs: (300, 5)
  attack  epsilon           model      RMSE  run
0  clean      NaN            base  0.115062    1
1  clean      NaN  morph_ensemble  0.191335    1
2    pgd      0.1            base  0.249135    1
3    pgd      0.1      morph_rand  0.193250    1
4    pgd      0.2            base  0.338785    1

[PGD | Clean] stats:
         model     mean  std      min       q1   median       q3      max  n
          base 0.115062  0.0 0.115062 0.115062 0.115062 0.115062 0.115062 30
morph_ensemble 0.191335  0.0 0.191335 0.191335 0.191335 0.191335 0.191335 30
Saved clean stats to: /content/drive/MyDrive/298/Elec/FinalPipeline/pgd/results/pgd_clean_stats.csv

[PGD | pgd] base stats:
 epsilon     mean      std      min       q1   median       q3      max  